# Same-Time Sky-to-Science Coefficient Transfer with a Physically Split Continuum

**Variant status.** This notebook is the split-zodi branch of the deployed
symmetric dual-encoder group-head regressor. The corpus has been re-decomposed
with `sky_decomp.SkyDecompLSFSurfaceIterative(split_zodi=True)`, and the ML
model learns to predict the resulting two-family continuum representation.
See the identifiability notebook
`notebooks/moon_zodi_split_identifiability.ipynb` for the decomposition-side
design and validation.

---

# Chapter 1 — Decompositions

## 1.1 Goal and pipeline

LVM's science fibres and its two sky arms (SKY_NEAR, SKY_FAR) sample three
lines of sight simultaneously. Each row (fibre × exposure) is fit
**independently** by the same physical model to yield a coefficient vector
$\mathbf{c}\in\mathbb{R}^{P}$ with $P=433$ (see §1.3). The fit is a
constrained quadratic program (QP) driven by the observed flux and its
inverse-variance plus curvature regularisers. The output corpus for this variant lives at
`skysub/moon_zodi_spline/` with suffix `_lsf_surface_iterative_split_zodi`.

The corpus we consume here is produced by

```
python skysub/decompose_parallel.py \
    skysub/moon_zodi_spline/lvmsframe_median_stack_1.2.1_p40_p70.fits \
    skysub/sky_decomp/data/palace \
    --fit-model lsf-surface-iterative-split-zodi \
    --moon-zodi-data-root skysub/sky_decomp/data \
    --palace-oh-suffix _joint_v2_updated \
    --palace-diffuse-suffix _joint_native_adam_invsky_p2_10000iter \
    --n-refinement-cycles 5 --n-workers 8 \
    --output-dir skysub/moon_zodi_spline
```

## 1.2 Physical model

The decomposition writes each row's spectrum $y(\lambda)$ as a non-negative
sum of physical templates:

$$
y(\lambda) \;\approx\; \sum_k c_k\; T_k(\lambda),\qquad c_k\ge 0,
$$

where $T_k$ is the LSF-convolved template of family $k$. All families share
the row's Gaussian LSF sigma $\sigma_{\rm lsf}(\lambda)$ (see §1.6).

### 1.2.1 Moon carrier (15 basis functions)

$$
T_{\rm moon,\,j}(\lambda) \;=\; I_\odot(\lambda)\; \alpha_{\rm ROLO}(\lambda,\,\phi_0{=}30°)\; B_j^{(K_{\rm m}=11)}(\lambda),
$$

with $I_\odot$ the solar SED (Meftah 2018 UV–NIR reference), $\alpha_{\rm ROLO}$
the disk-integrated lunar albedo at a fiducial phase angle, and
$B_j^{(K_{\rm m})}$ a cubic B-spline basis with 11 interior knots (15 columns) after
the spline3 knot-count reduction (2026-08-25; the pre-reduction default was
25 knots / 29 columns). Flux-fit quality on bright-moon and moon-down regimes
stays within 1–3% of the pre-reduction baseline while eliminating all
adjacent-knot |corr| > 0.99 pairs. The ROLO envelope carries sharp mineral
absorption bands, giving the moon carrier a distinctive spectral shape.

### 1.2.2 Zodi carrier (5 basis functions, **new** family)

$$
T_{\rm zodi,\,j}(\lambda) \;=\; I_\odot(\lambda)\; (\lambda/5000\,\text{\AA})^{0.26}\; B_j^{(K_{\rm z}=1)}(\lambda),
$$

with $K_{\rm z}=1$ interior knot (5 columns) after the spline3 reduction
(2026-08-25; the pre-reduction default was 3 knots / 7 columns) and $0.26$
the Leinert-style zodiacal reddening exponent. The 1-knot spline is heavily
curvature-penalised ($\lambda_{\rm z}=0.1$, §1.4) so the zodi carrier is
nearly monotone in $\lambda$. Introducing this family (`split_zodi=True`)
is the central decomposition-side change of this variant: pre-split, the
smooth zodi
component was silently absorbed into `Moon_bs` on moon-down + low-$|\beta_{\rm ecl}|$
rows, giving `c_moon` a mixed identity that the ML head could not learn.

### 1.2.3 OH Meinel bands

Rotationally-grouped OH transitions from the PMD `pmd_popmodel_OH` file
produce ~110 stick spectra $L_i(\lambda)$; these are convolved with the row
LSF and stacked into `matrix_oh`. Coefficients are `OH_ddd` (rotational
level).

### 1.2.4 Diffuse continuum templates

Three fixed diffuse templates $D_1,D_2,D_3$ (HO2, FeO, O2Ac) come from
`pmd_refcont`. They carry the flat, wavelength-smooth part of the continuum
so `Moon_bs` and `Zodi_bs` compete only for curvature-sensitive shape.

### 1.2.5 Molecular O2 and atomic lines

Two O2 stick libraries ($O_2$, $O_2^b$) and two atomic families (K I 7699,
N I 5199) round out the design matrix. Names: `O2_bd`, `ATOM_K`, `ATOM_N`.

## 1.3 Coefficient block and design matrix

Concatenating the templates gives the row-level design matrix
$\mathbf{A}\in\mathbb{R}^{N_\lambda\times P}$:

$$
\mathbf{A} \;=\; [\, \mathbf{A}_{\rm oh}\ |\ \mathbf{A}_{\rm moon}\ |\ \mathbf{A}_{\rm zodi}\ |\ \mathbf{A}_{\rm diff}\ |\ \mathbf{A}_{\rm atom}\ |\ \mathbf{A}_{\rm orc}\ |\ \mathbf{A}_{\rm o2}\,].
$$

For this variant $N_\lambda=12401$ pixels on 3600–9800 Å at $\Delta\lambda=0.5$ Å,
and $P=433$ coefficients (deployed spline3 reduction, 2026-08-25):

| block         | count | code family    | physical driver                           |
|:--------------|------:|:---------------|:------------------------------------------|
| OH Meinel     |   ~400| `OH_ddd`        | thermospheric temperature, geomagnetic $A_p$ |
| Moon spline   |    15 | `Moon_bs\d+`    | $g_{\rm moon}(\phi, h, r_{\rm sep})$      |
| Zodi spline   |     5 | `Zodi_bs\d+`    | $B_{500}(\lambda_\odot^{\rm rel}, |\beta_{\rm ecl}|)$·airmass |
| Diffuse       |     3 | HO2, FeO, O2Ac  | mesospheric chemistry residual            |
| Atomic K I    |     3 | `ATOM_K`        | K I 7699 nightglow                        |
| ORC / N I     |     4 | `ATOM_N`        | N I 5199 nightglow                        |
| Molecular O2  |     3 | `O2_bd`         | O2 Meinel                                 |

Deployed knot counts: `n_spline_knots=11` for moon and
`n_zodi_spline_knots=1` for zodi, auto-inferred from `coef_names` by the
`infer-spline-knots` notebook cell.

## 1.4 QP formulation

For each row, the fit solves a **weighted non-negative regularised
least-squares** problem:

$$
\boxed{\;\;\mathbf{c}^\star \;=\; \arg\min_{\mathbf{c}\ge 0}\;\; \underbrace{\tfrac{1}{2}\big\lVert \operatorname{diag}(\sqrt{\mathbf{w}})\big(\mathbf{A}\mathbf{c}-\mathbf{y}\big)\big\rVert_2^2}_{\text{data likelihood}} \;+\; \underbrace{\lambda_{\rm m}\lVert \mathbf{D}^{(2)}_{\rm m}\mathbf{c}_{\rm m}\rVert_2^2 \;+\;\lambda_{\rm z}\lVert \mathbf{D}^{(2)}_{\rm z}\mathbf{c}_{\rm z}\rVert_2^2}_{\text{spline curvature}}.\;\;}
$$

Symbols:

- $\mathbf{y}\in\mathbb{R}^{N_\lambda}$: the observed spectrum, scaled by
  `FACTOR = 1e14` (to bring counts to O(1)).
- $\mathbf{w} = 1/\sigma_{\rm pix}^2$: pixel inverse-variance from the
  reduction pipeline (`IVAR`, `SKY_IVAR`), with a relative floor
  `PIXEL_SIGMA_FLOOR_REL` per HDU.
- $\mathbf{D}^{(2)}$: second-difference operator on adjacent spline
  coefficients; suppresses ripples/oscillations in the moon and zodi splines.
- $\mathbf{c}\ge 0$: hard non-negativity constraint on every column
  (every family is physically non-negative).

The default curvature strengths (per unit scaled flux $y'=y/d$, see §1.5) are

$$
\lambda_{\rm m}=10^{-3},\qquad \lambda_{\rm z}=10\,\lambda_{\rm m}=10^{-1}.
$$

The heavier zodi penalty reflects that with only $K_z=3$ interior knots the
zodi spline should encode a smooth departure from the power-law, not sharp
features.

### 1.4.1 Solver

The problem is convex and is delegated to **Clarabel** (interior-point QP,
sparse triangular Hessian). The active-set structure at the optimum is used
downstream to compute coefficient uncertainties (§1.9).

### 1.4.2 Interline reweighting (optional)

The fit optionally reweights pixels near strong OH lines by
`moon_interline_weight ∈ [0, 1]` to prevent line residuals from stealing
signal from `Moon_bs`. This is disabled in the deployed split-zodi
configuration but the option remains in `fit.py`.

## 1.5 Linear algebra: rescaling for stability

Direct QP on physical units suffers from column-scale imbalance
(order-of-magnitude differences between OH sticks and B-spline columns).
`SkyDecomp._solve_nonnegative_weighted` applies a **two-stage rescaling**:

1. **Data scale.** Let $d = \max\big(\sqrt{\overline{y_w^2}},\,1\big)$
   where $y_w = \sqrt{\mathbf{w}}\odot\mathbf{y}$. Set $\mathbf{y}' = \mathbf{y}/d$.
2. **Column scale.** Let $s_k = \lVert (\sqrt{\mathbf{w}}\odot\mathbf{A})_k\rVert_2$
   per column. Set $\mathbf{A}'_{:,k} = \mathbf{A}_{:,k}/(d\,s_k)$.
3. Solve $\mathbf{c}'^\star = \arg\min_{\mathbf{c}'\ge 0} \tfrac{1}{2}\lVert\sqrt{\mathbf{w}}(\mathbf{A}'\mathbf{c}'-\mathbf{y}')\rVert^2 + \text{(rescaled regularisers)}$
   using Clarabel.
4. **Undo the rescaling:** $c_k = c'_k / s_k$.

The regularisation operators are rescaled consistently:

- Curvature: $\mathbf{D}^{(2)}_{\rm scl} = \mathbf{D}^{(2)}\,\operatorname{diag}(1/s_{\rm m})$
  (and similarly for zodi).

## 1.6 LSF-surface-iterative refinement loop

`SkyDecompLSFSurfaceIterative` alternates between:

1. **QP fit** of $\mathbf{c}^\star$ given the current LSF $\sigma_{\rm lsf}(\lambda)$
   and continuum baseline.
2. **LSF-surface update.** Extract a low-order surface in
   $(\lambda,\text{fibre})$ from the OH-line residual widths; refit
   $\sigma_{\rm lsf}$ per row.
3. **Continuum baseline update.** Small local update of the diffuse family
   from the low-pass residual.

The loop runs `n_refinement_cycles = 5` by default. Each cycle refits the
same design matrix $\mathbf{A}$ (rebuilt with the updated LSF); the QP is
warm-started from the previous solution. Convergence is monitored by the
relative change in `chi2_reduced`.

## 1.7 Weights

The data-side weights carry through the pipeline as inverse-variance:

- **Base ivar**: from `IVAR` / `SKY_IVAR` HDUs, floored by
  `PIXEL_SIGMA_FLOOR_REL · (typical pixel signal)` per HDU family
  (`PIXEL_SIGMA_HDU_NAMES`).
- **Interline boost** (optional): $w_i \to w_i\cdot m^{\rm il}(\lambda_i)$
  with $m^{\rm il}<1$ on strong OH cores; disabled here.
- **Finite mask**: only pixels with $y_i,\sigma_i,w_i$ all finite and
  $w_i>0$ enter the QP.

## 1.9 Coefficient uncertainties

At the QP optimum, split coefficients into active (interior, $c_k>0$) and
inactive (pinned at 0, $c_k=0$). On the active subset $\mathcal{A}$, the
regularised Fisher information is

$$
\mathbf{F}_{\mathcal{A}} \;=\; \mathbf{A}_{:,\mathcal{A}}^{\top}\operatorname{diag}(\mathbf{w})\mathbf{A}_{:,\mathcal{A}} \;+\; 2\lambda_{\rm m}\mathbf{D}^{(2)\top}\!\mathbf{D}^{(2)}\big|_{\mathcal{A}} \;+\; 2\lambda_{\rm z}\mathbf{D}^{(2)\top}\!\mathbf{D}^{(2)}\big|_{\mathcal{A}} \;+\; \ldots
$$

The covariance is $\mathbf{F}_{\mathcal{A}}^{-1}$, inflated by
$\max(\chi^2_\nu,\,1)$ to absorb model misfit. Column-scale and data-scale
are undone before reporting; inactive coefficients receive `NaN`. See
`SkyDecomp._coef_err_active_set`. These uncertainties feed the ML loss
weights via `COEF_ERR` (§3.6.2).

**Joint covariance persistence.** The diagonal $\boldsymbol\sigma_{\rm coef}=\sqrt{\operatorname{diag}(\mathbf{F}_{\mathcal{A}}^{-1})}$ discards all off-diagonal information; for a near-collinear basis (adjacent Moon_bs
knots stay at $|\rho|\!\sim\!0.88$ even at the reduced $K\!=\!11$), that
marginal $\sigma$ substantially over-states the constraint on any
individual knot while under-stating the constraint on any joint quantity
(block amplitude, PCA scores, LSF-convolved flux over the moon block).
`SkyDecomp._coef_err_active_set` therefore also materialises, on the same
solve-group Hessians, the FULL active-set covariance restricted to the
moon and (when `split_zodi=True`) zodi B-spline blocks:

$$
\mathbf{\Sigma}_g \;=\; \big[\mathbf{F}_{\mathcal{A}}^{-1}\big]_{\mathcal{A}\cap g,\;\mathcal{A}\cap g}\qquad g\in\{\text{moon},\text{zodi}\}
$$

with the same column-scale and per-column $\chi^2$ inflation applied
consistently on the diagonal and off-diagonal (so
$\sqrt{\operatorname{diag}(\mathbf{\Sigma}_g)}$ reproduces the entries
of $\boldsymbol\sigma_{\rm coef}$ in that block); inactive rows/columns
are `NaN`.  These matrices are persisted per row to disk as the
`COEF_COV_MOON` and `COEF_COV_ZODI` ImageHDUs by `results_to_fits`
(shape $n_{\rm row}\!\times\!n_g\!\times\!n_g$; skipped for the
degenerate $n_g\!=\!1$ physical moon-zodi model).  Per-row storage
overhead at deployed dimensions is 15×15+5×5 = 250 float64 per row
$\approx 2\,\text{kB}$; a full corpus adds $\sim$100 MB across the
three arms.  The compact `*_meta_coef_*.fits` files carry both HDUs
through `extract_meta_and_coef_products`.  See §3.6.2 for how the
notebook consumes them.


## 1.10 Input data

Each row's inputs to the fit are:

- $\mathbf{y}$: 12401-pixel median-stacked flux on 3600–9800 Å.
- $\mathbf{w}$: per-pixel inverse variance.
- $\sigma_{\rm lsf}(\lambda)$: LSF Gaussian sigma per pixel (updated
  iteratively).
- Row metadata: MJD (for solar-activity, ecliptic geometry), fibre RA/Dec
  (for airmass, moon separation, ecliptic latitude), exposure duration.

The corpus consumed here is
`skysub/moon_zodi_spline/lvmsframe_median_stack_1.2.1_p40_p70*.fits`, all
with suffix `_lsf_surface_iterative_split_zodi` and containing a new
`COMP_ZODI` HDU. The wavelength cache is
`moon_zodi_spline/coef_wavelengths_basis_v4.npz`.

---

# Chapter 2 — Geometric normalisation & effective extinction

**Framing principle.** The ML transfer (Chapter 3) is trained and evaluated
in **intrinsic-emissivity space**, not in observed-flux space. Each row's
decomposition coefficients (Chapter 1) are pre-divided by the row's own
line-of-sight geometry factor $V(z;h)\cdot 10^{-0.4 k(X-1)}$ **before** they
enter the network, so the network sees the zenith-equivalent amplitude of the
underlying emitting layer. On the output side, predictions are multiplied
back by the science-arm geometry factor to restore the **observed** amplitude
at the science pointing:

$$
\boxed{\;\;\mathbf{c}^{\rm intrinsic}_{r,g} \;=\; \frac{\mathbf{c}^{\rm obs}_{r,g}}{V(z_r;\,h_g)\cdot 10^{-0.4\,k_g\,(X_r-1)}} \;\;\xrightarrow{\;\text{ML}\;}\;\; \hat{\mathbf{c}}^{\rm intrinsic}_{{\rm sci},g} \;\;\xrightarrow{\;\times V(z_{\rm sci};h_g)\cdot 10^{-0.4\,k_g\,(X_{\rm sci}-1)}\;}\;\; \hat{\mathbf{c}}^{\rm obs}_{{\rm sci},g}.\;\;}
$$

The reason this framing works is that intrinsic emissivity is the quantity
that **actually transfers** from one line of sight to another: gravity-wave
fluctuations aside, the emitting layer at 87 km looks the same in whatever
direction you point during a 900-s exposure. Zenith angle, altitude and
extinction on the other hand differ between the two sky arms and the science
arm, and if the network had to learn those differences from context it would
be re-deriving well-known atmospheric physics on limited training data. By
dividing them out **in closed form before the network**, we let the ML head
focus on the remaining physics — gravity-wave and short-timescale variability
that the sky arms can genuinely constrain for the science arm.

The decomposition (Chapter 1) returns coefficients in **observed physical
units**: amplitudes at the actual pointing, at the actual zenith angle, at the
actual airmass. Chapter 2 is the closed-form transform that converts those to
the intrinsic emissivities the ML consumes, and back again on the output side.

## 2.1 The scaling law

For an airglow layer at zenith angle $z$ and target airmass $X(z)$, the
observed amplitude relates to the intrinsic (zenith-equivalent) emissivity by

$$
A_{\rm obs} \;=\; A_{\rm int}\;\underbrace{V(z;h)}_{\text{slant path}}\;\underbrace{10^{-0.4\,k(\lambda)\,[X(z)-1]}}_{\text{extinction}},
$$

with the **van Rhijn** slant-path factor for a thin shell at height $h$ above
Earth radius $R_\oplus$

$$
V(z;h)\;=\;\left[1-\left(\frac{R_\oplus}{R_\oplus+h}\right)^{\!2}\sin^2 z\right]^{-1/2},\qquad V(0;h)=1.
$$

$V$ **saturates** toward the horizon (6.0 at $h=87$ km, 3.4 at $h=285$ km)
rather than diverging like $\sec z$, because the observer sits inside a curved
shell. Using plane-parallel airmass instead overstates the enhancement by 4%
at $z=60°$ for a mesospheric layer and 12% for an ionospheric one — a smooth
function of $z$, so it does not average away and would alias into whatever the
network learns as geometry.

## 2.2 Where it is applied

`airglow_geometry_scale()` returns $V\!\cdot\!10^{-0.4k(X-1)}$ per row and per
coefficient, and is applied **in physical units, before the square-root /
asinh transforms and before robust scaling**. Neither of those operations
commutes with a division by the geometry factor: the scaler subtracts a
median, so $(\sqrt{c}-m)/V\ne\sqrt{c/V}-m$, and under the square root the
correct divisor would be $\sqrt{V}$ rather than $V$. Applying the correction
downstream of either transform produces an error of order $V$ itself.

The full training-time pipeline for each airglow group therefore is:

1. **Decompose** each row → observed coefficients $\mathbf{c}^{\rm obs}_{r,g}$.
2. **Divide out geometry** (this chapter):
   $\mathbf{c}^{\rm int}_{r,g} \;=\; \mathbf{c}^{\rm obs}_{r,g}\,/\,\big[V(z_r;h_g)\cdot 10^{-0.4k_g(X_r-1)}\big]$
   for both sky arms and the science arm, independently.
3. **Compress** (§3.3): elementwise transform + PCA truncation.
4. **Robust-scale** across the training set.
5. **ML forward** (§3.4): predict $\hat{\mathbf{s}}^{\rm int}_{\rm sci,g}$ (still in intrinsic-emissivity, robust-scaled, compressed space).

At inference the pipeline is inverted:

6. **Inverse robust scaling**, **inverse compressor**, then
7. **Multiply back by science-arm geometry**:
   $\hat{\mathbf{c}}^{\rm obs}_{{\rm sci},g} \;=\; \hat{\mathbf{c}}^{\rm int}_{{\rm sci},g}\cdot\big[V(z_{\rm sci};h_g)\cdot 10^{-0.4k_g(X_{\rm sci}-1)}\big]$,
   inside `predict_sci_coefficients_default()`.
8. **Reconstruct** the observed sci spectrum $\hat y_{\rm sci}(\lambda) = A\hat{\mathbf{c}}_{\rm sci}$.

**The ML target is therefore the intrinsic (zenith-equivalent) emissivity of
each layer, not the observed coefficient**. Two direct consequences:

- The three `vanrhijn_*` context columns are redundant as network inputs for
  the airglow groups (they are already absorbed by the pre-division) and can
  be dropped at the encoder boundary via `drop_vanrhijn_from_context=True`.
  `alt` and `airmass` are kept because the `moon` and `zodi` groups still
  need them.
- The **loss** is computed on intrinsic-emissivity residuals, not on observed
  residuals. Rows at high airmass therefore contribute to the loss with the
  same weight as rows at zenith, which is the physically correct behaviour:
  the underlying physics we are trying to interpolate does not know the
  target's zenith angle.

Because the reference point is $X=1$ rather than $X=0$, the recovered
"emissivity" is a **zenith-equivalent** amplitude, still carrying one airmass
of extinction. This is self-consistent and cancels in the sky-to-sci transfer,
but it is not a physical layer emissivity and should not be interpreted as
one.

`assert_context_is_physical()` guards every call. It checks physical ranges
and cross-checks any `vanrhijn_*` column against van Rhijn recomputed from
`alt`. Those two are computed independently upstream, so disagreement means
the context was transformed on the way in — the failure mode being guarded
against is evaluating geometry on scaler output, where an altitude of $\sim
0.5$ becomes a zenith angle of $\sim 89.5°$ and every row silently receives a
near-horizon factor of $\sim 6$.

## 2.3 Layer heights

Assigned per group via `GROUP_HEIGHT_FEATURE`:

| group          | van Rhijn feature | height       | physics                                         |
|:---------------|:------------------|-------------:|:------------------------------------------------|
| `mesospheric`  | `vanrhijn_87km`   | 87 km        | OH Meinel + O$_2$ atmospheric bands             |
| `atomic`       | `vanrhijn_95km`   | 95 km        | K, Na, mesopause metals / metastables           |
| `ionospheric`  | `vanrhijn_285km`  | 285 km       | O I 6300/6364 and F-region recombination lines  |
| `continuum`    | `vanrhijn_87km`   | 87 km        | HO$_2$ / FeO / O$_2$Ac mesopause chemiluminescence |
| `moon`         | (none)            | factor 1.0   | scattered moonlight — scattering geometry, not thin-shell emission |
| `zodi`         | (none)            | factor 1.0   | line-of-sight integrated interplanetary dust — not a thin shell     |

The `continuum` group name is retained from an earlier grouping in which
HO$_2$ / FeO / O$_2$Ac were treated as aerosol continuum; they are actually
airglow, but they are kept in a separate coefficient group from `mesospheric`
because they use a broadband basis rather than the OH line-group basis and
their compression pipeline stays sqrt-identity (§3.3). Moon and zodi
explicitly bypass van Rhijn: neither is thin-shell emission, and their
sky-to-sci transfer relies on other physics (Krisciunas–Schaefer geometry for
moon, Leinert lookup + airmass for zodi).

## 2.4 Per-coefficient effective wavelength

Extinction varies strongly across the LVM range, so a single wavelength per
group is not adequate: the `mesospheric` group alone spans O I 5577, Na D and
the OH / O$_2$ bands, and even within the OH Meinel system $k(\lambda)$ varies
by nearly a factor of two between the reddest and bluest bands.
`resolve_coef_wavelengths_a()` assigns a wavelength per coefficient, in this
order of precedence:

1. **basis centroid** — reconstructing with $\mathbf{c}=\mathbf{e}_j$ isolates
   basis component $j$, whose intensity-weighted centroid is
   $\lambda_{\rm eff}(j)=\sum\lambda|f_j|/\sum|f_j|$. This reads the actual
   basis, which is built from the same PMD population model, LSF and
   wavelength grid the decomposition uses at fit time, so the mesopause
   rotational Boltzmann factor for OH, the exact vibrational populations, and
   the blend structure across overlapping bands are all baked in by
   construction — it is the definitionally-correct weighting. Cached to
   `WAVELENGTH_CACHE` (`coef_wavelengths_basis_v4.npz` for the split-zodi
   basis).
2. **explicit wavelength token or species lookup** baked into `COEF_SCHEMA`
   for named species (K I 7699, N I 5199, Na D, O I 5577, O I 6300, O I 7774,
   O I 8446, O$_2$ b-band). Used only for the handful of coefficients whose
   basis support is empty on the current wavelength grid.
3. **group default** — `GROUP_EFFECTIVE_WAVELENGTH_A`, last resort.

Provenance is recorded per coefficient and printed for the airglow groups,
because a silently wrong wavelength becomes a silently wrong extinction
correction.

## 2.5 Effective extinction

The extinction coefficient $k(\lambda)$ enters as $10^{-0.4k(X-1)}$. Three
physical facts govern how it should be chosen, and the implementation follows
from them.

### 2.5.1 Only the ratio matters, so the absolute value barely does

The sky-to-sci transfer of an airglow amplitude is

$$
R\;=\;\frac{V(z_{\rm sci})}{V(z_{\rm sky})}\;10^{-0.4\,k\,(X_{\rm sci}-X_{\rm sky})},\qquad \frac{\partial\ln R}{\partial k}\;=\;-0.4\ln 10\;\Delta X\;\approx\;-0.921\,\Delta X.
$$

The absolute $k$ cancels; only $k$ times the airmass **difference** survives.
The three pointings are simultaneous and share one atmosphere, so a per-night
error $\delta k$ propagates as $0.921\,\delta k\,\Delta X$. With realistic
aerosol variability ($\delta k\sim 0.02$–$0.04$ mag/airmass) and $\Delta
X\sim 0.2$ that is a 0.2–0.7% transfer error, an order of magnitude below the
gravity-wave floor. **Per-observation extinction is therefore not modelled,
and does not need to be.**

### 2.5.2 A stellar curve is the wrong curve for an extended source

For a star, scattered photons are lost. Airglow is a quasi-uniform source
filling the sky, so photons scattered out of the beam are largely replaced by
photons scattered in from adjacent lines of sight, and the effective
attenuation is well below the single-scattering value. Scattering dominates
the total everywhere airglow matters — Rayleigh + aerosol is roughly 70–100%
of $k$ from 4000 Å to 1 µm at LCO's 2380 m — so a stellar curve
over-corrects, most severely in the blue. This is a **coherent bias**, not
random scatter.

### 2.5.3 $h$ and $k$ are nearly degenerate

Over the zenith range LVM observes, $\ln V(z;h)$ and $X-1$ are collinear to
$\rho>0.995$ for $z\le 60°$. A height error can masquerade as an extinction
error and vice versa. What the data constrain, and all the ML needs, is the
product $V\cdot 10^{-0.4 k(X-1)}$.

### 2.5.4 Implementation: fit $k_{\rm eff}$ from the airglow itself

Rather than model $k$, `fit_effective_extinction()` performs a **Bouguer fit
that uses airglow as its own source**. For coefficient $j$ and a simultaneous
sky pair,

$$
\ln\!\frac{A_{\rm near}}{A_{\rm far}}\;-\;\ln\!\frac{V_{\rm near}}{V_{\rm far}}\;=\;-0.4\ln 10\;k_{\rm eff}\,(X_{\rm near}-X_{\rm far})\;+\;\varepsilon,
$$

where $\varepsilon$ is the gravity-wave fluctuation between the two lines of
sight — zero mean in the log and uncorrelated with the airmass difference. The
intrinsic emissivity cancels because both arms are the same coefficient at the
same instant. The recovered $k_{\rm eff}$ **absorbs the multiple-scattering
correction, the airglow-versus-stellar difference, the site aerosol level and
any residual error in the assumed layer height**, none of which has to be
modelled explicitly.

Four details make the estimator trustworthy:

- **Layer height is held fixed.** Because of the degeneracy above, only $k_{\rm eff}$ is fitted. Fitting both would be ill-conditioned.
- **Selection is at column level only.** A per-row amplitude threshold is
  selection on the outcome: whichever arm sits at lower airmass has the
  smaller van Rhijn factor and fails the cut unless it carries a positive
  fluctuation, which correlates the retained residual with the sign of
  $\Delta X$. In testing, a 40th-percentile per-row cut inflated the
  recovered $k_{\rm eff}$ by a factor of 1.8. Coefficients are instead kept
  or dropped **whole**, via `min_positive_fraction`; `retained_frac` is
  reported so residual row-level loss is visible.
- **Errors are cluster-robust, clustered on rows.** All coefficients in a
  row share one gravity-wave fluctuation, so naive OLS errors are far too
  small.
- **Stability is reported.** Each bin carries split-half values, and the
  fitted intercept — which should be zero. A significantly nonzero intercept
  indicates a relative throughput offset between the two sky channels rather
  than anything atmospheric.

`resolve_coef_extinction_k()` then interpolates the fitted values in
wavelength over well-constrained bins, falls back to the generic LCO stellar
curve elsewhere, and **clips into $[0,k_{\rm generic}]$**: the multiple-
scattering argument makes the stellar curve an upper bound, so a fitted value
above it signals noise or an unmodelled gradient, not physics. The resulting
per-coefficient array is stored on the dataset as `coef_extinction_k`,
threaded into `airglow_geometry_scale`, and saved in the training artifacts so
that prediction uses exactly the values training used. Set
`USE_FITTED_EXTINCTION = False` to revert to the generic curve.

## 2.6 Moon and zodi are exempt

Neither the moon nor the zodi group is a thin-shell emitter, so
`airglow_geometry_scale()` returns factor 1.0 for their coefficient columns.
The physics they need is instead:

- **moon**: the Krisciunas–Schaefer atmospheric scattering geometry. The
  ML head consumes `moon_alt`, `moon_phase`, `moon_sep`, `airmass`
  directly and learns the full non-linear dependence.
- **zodi**: the Leinert V-band lookup $B_{500}(\lambda_\odot^{\rm rel},|\beta_{\rm ecl}|)$
  times airmass. The ML head consumes ecliptic geometry, `airmass` and
  (via the isolated zodi branch, §3.4.1) `vanrhijn_285km` as a smooth
  F-region-height proxy for the extended zodiacal geometry.

Extinction on moon and zodi is not fitted here either. Aerosol variability
enters the moon amplitude directly (Krisciunas–Schaefer $k_{\rm ext}$), but
the same per-night degeneracy with height-of-scatter argument applies: the
absolute value cancels between sky and sci, and only the airmass difference
survives, which is at the per-cent level.

---

# Chapter 3 — ML Reconstruction & Prediction

## 3.1 Goal

Given a row's paired coefficients from the two sky arms (near, far) and the
row's geometric / temporal context ("ctx"), predict the science-arm
coefficients $\hat{\mathbf{c}}_{\rm sci}$. Reconstructing
$\hat{y}_{\rm sci}(\lambda) = \mathbf{A}\hat{\mathbf{c}}_{\rm sci}$ then gives
the sky spectrum at the science pointing so that the science exposure can
be pixel-level sky-subtracted.

## 3.2 Coefficient grouping

Coefficients are routed into six physically meaningful groups by
`COEF_SCHEMA`:

| group          | patterns             | $n_{\rm coef}$ | dominant physics driver                        |
|:---------------|:---------------------|---------------:|:-----------------------------------------------|
| `mesospheric`  | `OH_\d{3}`, `O2_b\d+`|         ~403   | thermospheric temperature, geomagnetic activity |
| `moon`         | `Moon_bs\d+`         |             15 | $g_{\rm moon}(\phi, h, r_{\rm sep})$           |
| `zodi`         | `Zodi_bs\d+`         |              5 | $B_{500}(\lambda_\odot^{\rm rel}, |\beta_{\rm ecl}|)$ × airmass |
| `continuum`    | HO2, FeO, O2Ac       |              3 | mesospheric chemistry (residual)               |
| `atomic`       | ATOM_K               |              3 | night-side K I 7699                            |
| `ionospheric`  | ATOM_N               |              4 | night-glow N I 5199                            |

Deployed with the spline3 reduced basis (2026-08-25 changelog): $n_{\rm moon}=15$
from `n_spline_knots=11` and $n_{\rm zodi}=5$ from `n_zodi_spline_knots=1`. Total
$P = 433$ coefficients per row.

`zodi` is promoted to its own group by this variant. The rationale is physical:
$g_{\rm moon}$ and $B_{500}$ depend on disjoint context sets (lunar geometry vs.
ecliptic geometry) and the previous baseline's ML head had to reconcile the two
on one output. Splitting the head lets each learn the projection appropriate to
its physical support.

## 3.3 Per-group compressor

Each group's raw coefficient block $\mathbf{c}_g\in\mathbb{R}^{n_g}$ is projected
into a compact "score" vector $\mathbf{s}_g\in\mathbb{R}^{n_g^{\rm score}}$ via
a fitted compressor consisting of three stages:

1. **Robust per-column scale** $\boldsymbol\rho_g$ = per-column median of
   positive training values on the near arm.
2. **Elementwise forward transform** $\mathbf{u}_g = f_{\rm kind}(\mathbf{c}_g/\boldsymbol\rho_g)$
   with `kind` chosen per group by `COMPRESSION_TRANSFORM_BY_GROUP`. The deployed
   defaults are `asinh` for `mesospheric` (coefficients span many decades and
   include OH lines close to zero) and `sqrt` for every other group (`moon`,
   `zodi`, `continuum`, `atomic`, `ionospheric`).
3. **Standardise + PCA truncation** on the pooled (near + far + sci) training
   scores:
   $\mathbf{s}_g = \big((\mathbf{u}_g - \boldsymbol\mu_g)/\boldsymbol\sigma_g\big)\,\mathbf{B}_g\big[:,\mathcal{K}_g\big]$,
   where $\mathbf{B}_g$ is the PCA basis (columns sorted by variance) and
   $\mathcal{K}_g$ retains components with **cross-arm correlation** above
   `COMPRESSION_XARM_THRESHOLD` on the held-out set. Groups with
   `COMPRESSION_USE_PCA_BY_GROUP[g]=False` skip PCA (identity basis).

Compressed target dimensions used in this variant:

- `moon`: 15 → 15 (PCA rotation; the retention threshold keeps all 15
  components, so the rotation is applied but nothing is dropped)
- `zodi`: 5 → 5 (no PCA)
- `mesospheric`: 403 → 403 (no PCA)
- `continuum`: 3 → 3 (no PCA)
- `atomic`: 3 → 3 (no PCA)
- `ionospheric`: 4 → 4 (no PCA)

Total compressed score dim = 433, matching the uncompressed count.

Inverse compressor:

$$
\hat{\mathbf{c}}_g \;=\; f_{\rm kind}^{-1}\!\big(\hat{\mathbf{s}}_g\,\mathbf{B}_g[:,\mathcal{K}_g]^\top\!\cdot\!\boldsymbol\sigma_g + \boldsymbol\mu_g\big)\odot\boldsymbol\rho_g,
$$

with $f_{\rm kind}^{-1}\in\{{\rm identity},\,x\mapsto x^2,\,\sinh,\,\exp\}$
clipped to the per-column training range to prevent $\sinh/\exp$ blow-up on
outlier predictions.

## 3.4 Architecture: symmetric dual-encoder with group heads

`DualEncoderGroupHeadMLPCompressed` is the deployed network. All widths and
hyperparameters live in `default_dual_group_config`; the values shown below
are defaults for this variant.

Notation: $\mathbf{s}^{\rm nr},\mathbf{s}^{\rm fr}\in\mathbb{R}^{n_s}$
compressed scores on the two sky arms (concatenated across groups,
$n_s\!\approx\!450$ after RobustScaler-scaling); $\mathbf{x}^{\rm nr},\mathbf{x}^{\rm fr},\mathbf{x}^{\rm sc}\in\mathbb{R}^{n_x}$
row context features per arm (astropy geometry, VanRhijn heights, solar
activity, etc.).

**Step 1 — per-arm score encoder** (shared weights, applied to each sky arm):

$$
\mathbf{e}^{\rm nr} = \mathrm{MLP}_{\rm sky}\!\big([\mathbf{s}^{\rm nr};\,\mathbf{x}^{\rm nr}_{\rm model}]\big),\qquad
\mathbf{e}^{\rm fr} = \mathrm{MLP}_{\rm sky}\!\big([\mathbf{s}^{\rm fr};\,\mathbf{x}^{\rm fr}_{\rm model}]\big).
$$

Widths: `encoder_dims=(768, 384)` (default) with LayerNorm + GELU
between layers.

**Step 2 — context encoder** on the science-arm ctx only:

$$
\mathbf{e}^{\rm ctx} = \mathrm{MLP}_{\rm ctx}\!\big(\mathbf{x}^{\rm sc}_{\rm model}\big),\qquad \text{widths } (64,).
$$

**Step 3 — symmetric fusion.** Combine the two arm embeddings into
symmetric summaries so that swapping near ↔ far leaves the network
invariant:

$$
\mathbf{e}^{\rm mean} = \tfrac{1}{2}(\mathbf{e}^{\rm nr}+\mathbf{e}^{\rm fr}),\quad
\mathbf{e}^{\rm diff} = \mathbf{e}^{\rm nr}-\mathbf{e}^{\rm fr},\quad
\mathbf{e}^{\rm |diff|} = |\mathbf{e}^{\rm diff}|.
$$

Concatenate with the context embedding:
$\mathbf{z} = [\mathbf{e}^{\rm mean};\,\mathbf{e}^{\rm diff};\,\mathbf{e}^{\rm |diff|};\,\mathbf{e}^{\rm ctx}]$.
The signed `e_diff` term lets the trunk see the sign of any asymmetry
(near closer to moon than far, or vice-versa), while `|e_diff|` gives a
sign-agnostic magnitude that the trunk can gate on regardless of orientation.

**Step 4 — shared trunk MLP**:

$$
\mathbf{h} = \mathrm{MLP}_{\rm trunk}(\mathbf{z}),\qquad \text{widths } (320, 160).
$$

**Step 5 — per-group heads.** Each group $g$ has its own head that produces
a predicted signed **score vector** $\hat{\mathbf{s}}^{\rm head}_g\in\mathbb{R}^{n_g^{\rm score}}$:

$$
\hat{\mathbf{s}}^{\rm head}_g = \mathrm{MLP}_{\rm head,\,g}(\mathbf{h}),\qquad \text{widths } (\text{head\_dim}=192,\, n_g^{\rm score}).
$$

Heads emit **linear** outputs (no Softplus) in scaled-score space; the
non-negativity of physical coefficients is recovered later by the inverse
compressor's clipping.

### 3.4.1 Isolated zodi branch (Phase B, 2026-08-22b)

When `default_dual_group_config['zodi_ctx_restriction']` is a non-empty tuple
of ctx feature names, the zodi head **bypasses the shared trunk** and consumes
only:

- the **per-arm slice** of the restricted ctx features (near + far + sci
  concatenated, so the branch sees the local `zodi_log10_v` gradient and any
  per-arm moon geometry),
- the near and far zodi score blocks (5 dims each in the deployed reduced-basis
  variant; see §1.4 / spline3).

Routed through a small two-layer MLP `zodi_branch: 64→32 (GELU)` followed by a
`(32,)`-hidden head (Phase C, 2026-08-26d) emitting $n_{\rm zodi}^{\rm score}=5$
signed scores that match the 5 `Zodi_bs` knots after the spline3 basis reduction.

The deployed default restriction is 15 features:

```
('airmass', 'vanrhijn_285km',
 'ecl_beta_deg', 'ecl_lon_sin', 'ecl_lon_cos',
 'zodi_log10_v', 'sun_sep',
 'moon_alt', 'moon_sep', 'moon_phase_sin', 'moon_phase_cos',
 'moon_fli', 'moon_up_smooth', 'moon_airmass_up', 'moon_signal_proxy')
```

giving $15 \times 3 + 2 \times 5 = 55$ input dims. Features cover two physical
roles:

1. **Zodi physics** &mdash; `ecl_beta_deg`, `ecl_lon_{sin,cos}`, `zodi_log10_v`,
   `sun_sep`, `airmass`, `vanrhijn_285km`. The Leinert lookup (§1.2.2) plus
   airmass (via the F-region-height van Rhijn as a smooth zodiacal-geometry
   proxy) fully determines the intrinsic zodiacal amplitude.
2. **Moon-driven contamination regime** &mdash; `moon_alt`, `moon_sep`,
   `moon_phase_{sin,cos}`, plus the four physics-prior moon-scatter proxies
   `moon_fli`, `moon_up_smooth`, `moon_airmass_up`, `moon_signal_proxy` (§2.6).
   On moon-down rows the `Zodi_bs` coefficients carry whatever residual
   continuum shape the `split_zodi` decomposition could not place elsewhere,
   so the sky↔sci transfer depends on moon geometry even though the underlying
   zodiacal light does not. Adding the moon proxies lets the branch condition
   on "moon down" / "Q4 phase" / "bright moon close-in" regimes; combined with
   the per-regime Jensen lift (§3.6.6) the residual zodi bias on `moon_alt ≤ 0`
   rows fell from 7.7% to ~2% across the Phase A/B/D deployments.

The previous `sun_alt` entry (twilight regime, 2026-08-24b) was dropped when
the physics-prior augment absorbed its role via `moon_signal_proxy`, which
already vanishes when the moon is down and lets the moon-brightness features
carry the twilight-residual signal.

Bypassing the trunk cuts moon-driven and geomagnetic-driven leakage into the
zodi head at the cost of the trunk's richer representation. Empirically this
trade-off wins for zodi calibration on high-$|\beta_{\rm ecl}|$ / bright-moon
rows without hurting overall mean_eRMSE.

### 3.4.2 Isolated continuum branch (Phase B/D, 2026-08-26d/e)

Mirroring the zodi pattern, when `continuum_ctx_restriction` is a non-empty
tuple the continuum head **bypasses the shared trunk** through its own branch.
The trigger for this was a specific empirical finding: the
`residual_ctx_attribution` diagnostic (5-fold CV random-forest fit of per-row
per-group residual RMS on ctx) surfaced continuum with rf_R² = 0.70 driven
entirely by moon geometry — a non-linear coupling the shared 160-d trunk was
under-parameterised to represent.

The deployed default restriction is 9 features:

```
('moon_alt', 'moon_sep', 'moon_phase_sin', 'moon_phase_cos',
 'moon_fli', 'airmass',
 'moon_fli_x_phase_cos', 'moon_sig_x_lon_cos', 'moon_sig_x_lon_sin')
```

The last three are Phase-A' explicit interaction features (§3.4.5):
`moon_fli * moon_phase_cos` (2nd-order phase term) and
`moon_signal_proxy * ecl_lon_{cos,sin}` (moon-scatter × zodi anisotropy).

Branch: `continuum_branch: 128→64 (GELU)` (widened from `64→32` by Phase D on
2026-08-26e — the only group with real headroom above the sky-arm noise floor
got the extra capacity), followed by a `(64,)`-hidden head emitting
$n_{\rm continuum}^{\rm score}=3$ scores. Input dim: $9 \times 3 + 2 \times 3
= 33$. Isolated-branch parameter cost: ~16k.

### 3.4.3 Additive moon-zodi coupling (Phase F, 2026-08-27c)

Both moon (via the shared-trunk head) and zodi (via the isolated branch)
benefit from knowing the *joint* moon-scatter + zodi-geometry state — scattered
moonlight adds to the diffuse ecliptic background that the split-zodi
decomposition tries to route entirely into `Zodi_bs`. To let the two heads
*share* that joint representation without breaking zodi's Phase-B isolation,
an **additive coupling latent** is threaded into both head outputs.

When `moon_zodi_ctx_restriction` is a non-empty tuple and
`moon_zodi_mode="additive"` (deployed default), the model builds:

- a small `moon_zodi_coupling_branch: 94→64→32 (GELU)` (~8.4k params),
- a **zero-initialised** projector
  $P_{\rm moon} = \mathrm{Linear}(32,\, n_{\rm moon}^{\rm score}=15)$ (495 params),
- a **zero-initialised** projector
  $P_{\rm zodi} = \mathrm{Linear}(32,\, n_{\rm zodi}^{\rm score}=5)$ (165 params).

The branch consumes the union of moon-scatter + zodi-geometry ctx: 18 features
$\times$ 3 arms plus 2 arms $\times (n_{\rm moon}+n_{\rm zodi}) = 20$ score
dims, for 94 input dims total.

The forward pass then becomes (for $g \in \{{\rm moon},\,{\rm zodi}\}$):

$$
\hat{\mathbf{s}}_g \;=\; \alpha_g\,\mathbf{s}^{\rm nr}_g + (1-\alpha_g)\,\mathbf{s}^{\rm fr}_g \;+\; \hat{\mathbf{s}}^{\rm head}_g \;+\; P_g\bigl(\mathrm{CouplingBranch}(x_{\rm mz})\bigr),
$$

where $\hat{\mathbf{s}}^{\rm head}_g$ is the moon shared-trunk head output
$\mathrm{head}_{\rm moon}(\mathbf{h})$ for $g={\rm moon}$ and the isolated zodi
branch output $\mathrm{ZodiHead}(\mathrm{ZodiBranch}(x_{\rm zodi}))$ for
$g={\rm zodi}$. Because the projectors are initialised to output exactly zero
(both weights and bias zero), **training starts byte-identical to the
pre-Phase-F configuration**; the projectors learn what fraction of the
coupling latent to route into each head's residual.

The `moon_zodi_ctx_restriction` 18-feature union is the superset of the zodi
restriction (§3.4.1) plus the 3 Phase-A' interaction features:

```
('airmass', 'vanrhijn_285km',
 'ecl_beta_deg', 'ecl_lon_sin', 'ecl_lon_cos',
 'zodi_log10_v', 'sun_sep',
 'moon_alt', 'moon_sep', 'moon_phase_sin', 'moon_phase_cos',
 'moon_fli', 'moon_up_smooth', 'moon_airmass_up', 'moon_signal_proxy',
 'moon_fli_x_phase_cos', 'moon_sig_x_lon_cos', 'moon_sig_x_lon_sin')
```

**Alternate mode**: `moon_zodi_mode="shared_branch"` (Phase E, 2026-08-27b,
tried and reverted) retires both the shared-trunk moon head AND the isolated
zodi branch and routes moon and zodi through a single larger branch. It
captured a bigger moon improvement (~7% moon_up sky-arm) but reintroduced the
Phase-B moon→zodi contamination and regressed atlas mid-band RMS by 35%. Kept
behind the `moon_zodi_mode` selector for future A/B; the deployed default is
`"additive"`. To fully revert either mode to the pre-coupling baseline, set
`moon_zodi_ctx_restriction=None`.

### 3.4.4 Data-flow diagram of the augmented model

```
    near-arm scores   far-arm scores   near-ctx (39)   far-ctx (39)   sci-ctx (39)
          |                 |               |               |             |
          +--- concat ------+               v               v             |
          |    scores/ctx                   |               |             |
          v                                 |               |             |
    +-----------+                       +----------------+                |
    | sky enc.  |                       |  sky encoder   |                |
    | (shared)  |------e_near-----+     |   (shared)     |------e_far     |
    +-----------+                 |     +----------------+          |     |
                                  v                                 v     v
                        e_mean = (e_near+e_far)/2  e_diff = e_near-e_far  |
                        |e_diff|                                          |
                        |                                                 |
                        v                                                 v
                        +---------- concat with e_ctx ---------------------+
                                          |
                                          v ctx encoder(sci) -> e_ctx
                                          |
                                    +-----------+
                                    |   trunk   |  (320 -> 160)
                                    +-----+-----+
                                          |
                                          v h (160-d)
                             +--------+---+---+--------+-----------------+
                             |        |       |        |                 |
                             v        v       v        v                 |
                         moon head  meso    iono     atomic              |
                                    head    head     head                |
                                    (shared trunk consumers)             |
                                                                         |
                        +------------------------------------------------+
                        |                                                |
                        v                                                v
                +---------------+                              +----------------+
                | zodi branch   |  (55-d in)                   | continuum br.  |
                | 64 -> 32      |                              | 128 -> 64      |
                | + zodi head   |                              | + cont. head   |
                | (32,) + n_z=5 |                              | (64,) + n_c=3  |
                +-------+-------+                              +--------+-------+
                        |                                              |
        (Phase F additive coupling latent, applied to moon + zodi)      |
        +----------------+                                              |
        |                                                               |
        v                                                               v
+-------------------+                                                +----+
| coupling branch   |                                                |    |
| 94 -> 64 -> 32    |                                                |    |
+-------+-----------+                                                |    |
        |                                                            |    |
        +--> P_moon = Linear(32, 15), zero-init  --> + moon head      |    |
        +--> P_zodi = Linear(32,  5), zero-init  --> + zodi head      |    |
                                                                     v    |
                                                              +---+ +---+ +---+ +---+ +---+
                                                     out[g] = |bl.|+|hd |+|D_g|  for each  g
                                                              +---+ +---+ +---+     of six
                                                    where D_g = coupling residual (moon,
                                                    zodi only) or 0 for the other four.

Blend for group g (see §3.5):
    hat{s}_g = alpha_g * s_g_near + (1 - alpha_g) * s_g_far + head_g + P_g(coupling)
    alpha_g is scalar OR per-row sigmoid(w.x_sci + b) for anisotropic
    groups (moon, zodi, continuum) via the ctx-alpha predictor.
```

The `CouplingBranch` runs once per batch and its 32-d latent is projected
independently into moon and zodi; the isolated zodi branch continues to run
unchanged in parallel (the two zodi contributions sum).

### 3.4.5 Context feature routing

The context vector is **39 features per arm** after the two augment stages
(ECLIPTIC-CTX-V1 and PHYSICS-PRIORS-V1). Every feature enters the sky encoder
(per-arm) and the ctx encoder (sci-arm), so the shared trunk sees the full
ctx. The specialised branches and the ctx-α predictor take restricted slices.

**Feature categories** (39 total):

| category | count | features |
|---|---:|---|
| Sci-pointing astrometry | 4 | `alt`, `az_sin`, `az_cos`, `airmass` |
| Moon geometry | 6 | `moon_alt`, `moon_sep`, `moon_phase_{sin,cos}`, `moon_az_{sin,cos}` |
| Sun geometry | 4 | `sun_sep`, `sun_alt`, `sun_az_{sin,cos}` |
| Sci-arm separation | 1 | `sci_sep` |
| Van Rhijn slant-path factors | 3 | `vanrhijn_87km`, `vanrhijn_95km`, `vanrhijn_285km` |
| Cyclic time features | 6 | `obstime_{day,lunation,year}_{sin,cos}` |
| Space-weather activity | 4 | `f107`, `f107_81d`, `kp`, `ew` |
| Ecliptic geometry (ECLIPTIC-CTX-V1) | 3 | `ecl_beta_deg`, `ecl_lon_{sin,cos}` |
| Physics-prior moon-scatter proxies (PHYSICS-PRIORS-V1) | 5 | `zodi_log10_v`, `moon_fli`, `moon_up_smooth`, `moon_airmass_up`, `moon_signal_proxy` |
| Phase-A' explicit interaction features | 3 | `moon_fli_x_phase_cos`, `moon_sig_x_lon_{cos,sin}` |

**Feature-to-branch routing** (✓ = feature enters that branch as one of its
ctx dims; empty = does not). The **shared trunk** column stays implicit —
every feature feeds the trunk via the encoders.

| feature | isolated zodi (§3.4.1) | isolated continuum (§3.4.2) | moon-zodi coupling (§3.4.3) | ctx-α (§3.5) |
|---|:---:|:---:|:---:|:---:|
| `alt`, `az_sin`, `az_cos` | | | | |
| `airmass` | ✓ | ✓ | ✓ | ✓ |
| `moon_alt`, `moon_sep` | ✓ | ✓ | ✓ | |
| `moon_phase_sin`, `moon_phase_cos` | ✓ | ✓ | ✓ | |
| `moon_az_sin`, `moon_az_cos` | | | | |
| `sun_alt`, `sun_az_{sin,cos}` | | | | |
| `sun_sep` | ✓ | | ✓ | |
| `sci_sep` | | | | |
| `vanrhijn_87km`, `vanrhijn_95km` | | | | |
| `vanrhijn_285km` | ✓ | | ✓ | |
| `obstime_{day,lunation,year}_{sin,cos}` | | | | |
| `f107`, `f107_81d`, `kp`, `ew` | | | | |
| `ecl_beta_deg` | ✓ | | ✓ | ✓ |
| `ecl_lon_sin`, `ecl_lon_cos` | ✓ | | ✓ | |
| `zodi_log10_v` | ✓ | | ✓ | |
| `moon_fli` | ✓ | ✓ | ✓ | |
| `moon_up_smooth` | ✓ | | ✓ | ✓ |
| `moon_airmass_up`, `moon_signal_proxy` | ✓ | | ✓ | |
| `moon_fli_x_phase_cos` | | ✓ | ✓ | |
| `moon_sig_x_lon_cos`, `moon_sig_x_lon_sin` | | ✓ | ✓ | |

**Head-to-branch routing** (which head consumes which upstream output):

| head | shared trunk | isolated zodi | isolated continuum | moon-zodi coupling residual |
|---|:---:|:---:|:---:|:---:|
| `moon` | ✓ | | | ✓ (additive) |
| `zodi` | | ✓ | | ✓ (additive) |
| `continuum` | | | ✓ | |
| `mesospheric` | ✓ | | | |
| `ionospheric` | ✓ | | | |
| `atomic` | ✓ | | | |

The specialised branches were added when the diagnostics pointed at
regime-specific structure the shared trunk was under-fitting; the additive
coupling was added to let moon share the zodi branch's ctx-restricted
representation without breaking zodi's isolation. See the changelog entries
for the empirical wins each branch bought.

## 3.5 Blend heads

Each group carries a **blend weight** $\alpha_g \in (0, 1)$ that mixes the
near/far arm scores with the head's residual prediction:

$$
\boxed{\;\hat{\mathbf{s}}_g \;=\; \alpha_g\,\mathbf{s}^{\rm nr}_g + (1-\alpha_g)\,\mathbf{s}^{\rm fr}_g \;+\; \hat{\mathbf{s}}^{\rm head}_g \;+\; \Delta_g^{\rm coupling}.\;}
$$

The additive coupling residual $\Delta_g^{\rm coupling}$ (§3.4.3) is non-zero
only for $g \in \{{\rm moon},\,{\rm zodi}\}$ when the coupling branch is
active; it is exactly zero at initialisation and remains identically zero for
the other four groups.

**Two parametrisations for $\alpha_g$:**

1. **Scalar α** (fallback): $\alpha_g$ stored directly (not through a
   sigmoid) as a single learnable scalar per group, clamped to
   $[\varepsilon,\, 1-\varepsilon]$ with $\varepsilon = 10^{-3}$ after each
   optimiser step. This is what the physically isotropic groups still use.

2. **Context-dependent α** (Phase A, 2026-08-26d, deployed for the
   anisotropic groups): when `alpha_ctx_features` is a non-empty tuple of
   ctx feature names, each group listed in `alpha_ctx_groups` gets a
   **per-row per-group** linear predictor
   $$\alpha_g(x_{\rm sci}) \;=\; \sigma\bigl(\mathbf{w}_g \cdot \mathbf{x}_{\rm sci}^{\alpha} + b_g\bigr),$$
   where $\mathbf{x}_{\rm sci}^{\alpha}$ is the sci-arm slice of the requested
   features. The linear predictor's weights are initialised to zero and its
   bias to $\mathrm{logit}(\alpha_{\rm init})$ so that at $t=0$ every row
   gets $\alpha_g = \alpha_{\rm init}$ (default 0.7), preserving byte-identity
   with the scalar-α starting point.

**Deployed defaults:**

- `alpha_ctx_features = ('moon_up_smooth', 'ecl_beta_deg', 'airmass')` — three
  features that carry moon-brightness, zodi-anisotropy, and airmass geometry,
  so the anisotropic groups can learn distinct blend rules per regime.
- `alpha_ctx_groups = ('moon', 'zodi', 'continuum')` — restricted to the
  physically anisotropic groups. The LOS-integrated thin-shell groups
  (`mesospheric`, `ionospheric`, `atomic`) keep the scalar α near 0.5: their
  sky↔sci transfer is genuinely isotropic on the LVM scale, and an early
  experiment applying ctx-α to all six groups blew up ionospheric sRMSE by
  23% before 26d restricted it.

**Physically-expected attractors during training:**

- `moon`: **anisotropic** on the LVM scale — the sky arm on the same side of
  the moon as the science fibre is closer in moon-projected geometry; the
  ctx-α predictor learns to drift toward 0.75–0.9 on bright-moon-close rows
  and stay near 0.5 on moon-down rows.
- `zodi`: **isotropic** on the LVM scale — both sky arms sit within a few
  arcminutes of the science field, well inside the zodiacal correlation
  scale; ctx-α stays near 0.5 across regimes. The isolated zodi branch
  (§3.4.1) plus the coupling latent (§3.4.3) do the physics work; ctx-α
  mostly reproduces the scalar-0.5 baseline for zodi.
- `mesospheric`, `atomic`, `ionospheric`: near 0.5 (LOS-integrated emissive
  layers using the scalar-α fallback).

Optionally (`moon_alt_conditional_alpha=True`) the moon blend can be split
into two learned scalars for $h_{\rm moon} \gtrless 0$; disabled in the
default deployed config since ctx-α subsumes it via the `moon_up_smooth`
feature.

## 3.6 Loss

The training loss is a **mixed-space** objective. The moon and zodi groups are
trained in **flux space** — the network's predicted compressed scores are
decompressed all the way to per-pixel spectra inside the loss and compared to
the target flux — while the other four groups (continuum, mesospheric,
ionospheric, atomic) stay in **scaled-score space** so that RobustScaler-
normalised residuals are comparable across compressed groups of very different
physical amplitudes. §3.6.1 defines the flux-space branch. §3.6.2 defines the
score-space branch. §3.6.3 covers the per-element COEF_ERR weighting.
§3.6.4 the diagnostic-only pixel WRMSE. §3.6.5 the row-weight boosts.
§3.6.6 the post-training per-coef Jensen-style lift.

### 3.6.1 Per-pixel flux MSE on moon and zodi (deployed)

For each group $g \in$ `flux_mse_groups = ('moon', 'zodi')` (deployed default)
the training loss is computed **directly on the reconstructed per-pixel
spectra**, not on the compressed score residuals. The motivation is that the
diagonal COEF_ERR-weighted score loss (§3.6.2 + §3.6.3) puts per-row weight
$\propto 1/\sigma^2$, and the truth-conditioned $|z|$ diagnostic (§1.9) shows
$\sigma_{\rm scaled}$ is essentially flat across the moon-brightness axis:
bright-moon rows (with $|c_{\rm moon}| \sim 10$–20, where flux fit errors
dominate the deliverable sky subtraction) end up on the same per-row loss
scale as moon-down rows (with $|c_{\rm moon}| \sim 0.05$). A flux-space MSE
couples per-row weight to target amplitude directly: the same absolute
score-space error contributes $\sim (|c_{\rm bright}|/|c_{\rm dim}|)^2 \sim
10^5$ times more from a bright row than from a dim row, matching the ratio
at which they matter in flux.

**Inverse compressor in torch.** For each configured group $g$ the trainer
maintains a differentiable torch implementation of `inverse_group_compressor`
(§3.3):

1. Undo the RobustScaler on the group's compressed score
   $\hat{\mathbf{s}}_g^{\rm scaled}$:
   $\hat{\mathbf{s}}_g^{\rm raw} = \hat{\mathbf{s}}_g^{\rm scaled}\odot \mathrm{scale}_g + \mathrm{center}_g$.
2. Undo the group's PCA rotation:
   $\mathbf{z}_g = \hat{\mathbf{s}}_g^{\rm raw} \, B_g^\top$
   ($B_g$ = identity when `use_pca=False`, e.g. zodi; a small orthogonal
   matrix stored on device when `use_pca=True`, e.g. moon).
3. Destandardise to forward-transformed space:
   $\mathbf{y}_g = \mathbf{z}_g \odot \mathrm{sd\_vec}_g + \mathrm{mean\_vec}_g$.
4. Invert the per-element transform. Both moon and zodi use `kind='sqrt'` so
   $\mathbf{em}_g = \max(\mathbf{y}_g, 0)^2$
   (sign is intentionally lost by the forward $\sqrt{}$; both target and
   prediction go through the same clamp so gradients are consistent).
5. Restore per-row geometry:
   $\hat{\mathbf{c}}_g^{(r)} = \mathbf{em}_g \odot \mathbf{sc}_{\rm sci}^{(r)}$
   where $\mathbf{sc}_{\rm sci}^{(r)}$ is the per-row per-coefficient geometry
   factor from `airglow_geometry_scale(ctx_sci, ...)` (see §2).

The identical chain is applied to the target scaled-score
$\mathbf{s}_g^{{\rm true},{\rm scaled}}$ so that both paths are in the same
physical units.

**Flux basis matrix.** A row-independent basis matrix
$A_g \in \mathbb{R}^{n_g^{\rm coef} \times n_\lambda^{\rm ds}}$
is precomputed once, before the ensemble seed loop, by instantiating

```python
SkyDecompLSFSurfaceIterative(
    wave_ref, lsf_sigma=1.0, n_spline_knots=N_MOON_KNOTS,
    split_zodi=SPLIT_ZODI, n_zodi_spline_knots=N_ZODI_KNOTS,
    base_dir=_infer_base_dir_for_reconstruction(),
    palace_oh_suffix='_joint_v2_updated',
    palace_diffuse_suffix='_joint_native_adam_invsky_p2_10000iter',
)
```

on the notebook's wavelength grid. The `SkyDecomp.__init__` build path (§1.2)
populates `matrix_moon` and `matrix_zodi` — the pre-LSF spline template
matrices $B_k(\lambda) \cdot \mathrm{solar\_rb}(\lambda) \cdot \mathrm{envelope}_g(\lambda)$,
one row per moon or zodi knot. The wavelength axis is decimated with stride 5
(a smoothly-varying continuum basis is invariant to this;
$n_\lambda^{\rm ds} \approx 2\,500$) which brings the per-batch matmul into a
comfortable GPU footprint while keeping the moon-albedo mineral bands and the
zodi power-law tilt at full resolution.

**Per-row loss.** With $A_g$ on device, the per-row per-group loss is

$$
\mathcal{L}_g^{\rm flux,(r)} \;=\; s_g \cdot \frac{1}{n_\lambda^{\rm ds}} \sum_{\lambda=1}^{n_\lambda^{\rm ds}} \Big( \hat{\mathbf{c}}_g^{(r)} A_g - \mathbf{c}_g^{{\rm true},(r)} A_g \Big)^2_\lambda
$$

and the group is added into the total loss with the same balancing weight
$w_g = m_g / \sqrt{n_g^{\rm score}}$ (§3.6.2) as the score-space groups, so
`moon_group_weight` and `zodi_group_weight` retain the calibrated meaning they
had under the score-space loss.

The per-group **scale-match factor**

$$
s_g \;=\; \frac{\text{median}_{r \in \text{train}}\; L_g^{\rm diag,(r)}}{\text{median}_{r \in \text{train}}\; \overline{f_g^{{\rm true},(r)}(\lambda)^2}}
$$

is computed once at trainer entry: it aligns the median per-row flux-MSE with
the median per-row diagonal-Huber loss (§3.6.2), so `moon_group_weight` and
`zodi_group_weight` keep the balancing meaning they would have under the
score-space form.

### 3.6.2 Weighted MSE on the score-space groups

For $g \in$ {`continuum`, `mesospheric`, `ionospheric`, `atomic`}, let
$\hat{\mathbf{s}}_g^{(r)},\,\mathbf{s}_g^{{\rm true},(r)}\in\mathbb{R}^{n_g^{\rm score}}$
be the predicted and target scaled-scores for row $r$ and group $g$. The
per-row per-group loss is

$$
\mathcal{L}_g^{(r)} \;=\; \frac{w_g}{n_g^{\rm score}}\,\sum_{k=1}^{n_g^{\rm score}} w^{\rm pe}_{g,k,r}\,\big(\hat{s}_{g,k,r} - s^{\rm true}_{g,k,r}\big)^2,
$$

with $w_g$ the **per-group balancing weight**

$$
w_g \;=\; \frac{m_g}{\sqrt{n_g^{\rm score}}},
$$

where the multipliers $m_g$ are the current deployed defaults:

- $m_{\rm moon}=2.0,\; m_{\rm zodi}=2.0,\; m_{\rm continuum}=1.0,\; m_{\rm mesospheric}=1.0,\; m_{\rm ionospheric}=1.0$.

The $1/\sqrt{n_g^{\rm score}}$ factor makes the sum-over-$k$ RMS-scale
comparable across compressed groups with different score dimensionality. Moon
and zodi carry $m_g=2.0$ multipliers even though their loss is computed in
flux space (§3.6.1) — the scale-match factor ports the multiplier meaning
across.

### 3.6.3 Heteroscedastic per-element weights

The decomposition-side coefficient uncertainties $\boldsymbol\sigma_{\rm coef,sci}$
(§1.9) are propagated through the compressor with a first-order Jacobian:

$$
\sigma_{s_{g,k,r}} \;=\; \big\lVert \nabla_{c_g} s_{g,k} \big\rVert \cdot \sigma_{{\rm coef},g,r},
$$

divided by the RobustScaler column scale so the weights live in the same space
as $\hat{\mathbf{s}}_g$. The per-element weight is

$$
w^{\rm pe}_{g,k,r} \;=\; \frac{1}{\sigma_{s_{g,k,r}}^2},\qquad \sigma_{s_{g,k,r}} \ge \sigma^{\rm floor}_{g,k},
$$

with $\sigma^{\rm floor}_{g,k}$ = `coef_err_sigma_floor_rel[g]` × per-column
median finite sigma. The floors are 5% for moon, zodi, continuum, and
ionospheric; 20% for mesospheric and atomic, because their $p_{99}/p_{50}$
sigma ratios span $10^{3}$–$10^{6}$ (see §7 diagnostic). Weights are
normalised so $\mathbb{E}_{\rm train}[w^{\rm pe}]=1$ per column; missing/
boundary sigmas fall through to the floor.

**Joint covariance diagnostic path.** The `coef-cov-loader` notebook cell
opens each of the three decomp FITS products, reads the `COEF_COV_MOON` and
`COEF_COV_ZODI` HDUs when present, slices them by `filtered_triplet['row_index']`,
and attaches `coef_cov_{moon,zodi}_{near,far,sci}` to `filtered_triplet`.
Two diagnostic cells consume these:

- `truth_conditioned_sigma_calibration` — computes the Mahalanobis residual
  $|z|_{\rm joint,g} = \sqrt{\mathbf{r}_g^{\top}\mathbf{\Sigma}_g^{-1}\mathbf{r}_g / n_{\rm active}}$
  per test row and reports its median vs the calibrated target
  $\sim\sqrt{\text{med}(\chi^2_n)/n}\approx 0.7$.
- `moon_sigma_investigation` — reports the joint per-block SNR
  $\sqrt{\mathbf{c}_g^{\top}\mathbf{\Sigma}_g^{-1}\mathbf{c}_g / n_{\rm active}}$
  binned by moon regime, alongside the independence-assuming
  $\lVert\mathbf{c}_g\rVert / \sqrt{\sum_j\sigma_j^2}$ for direct comparison.

These are diagnostic-only: the deployed loss uses only the diagonal
per-element $w^{\rm pe}$ weight above.

### 3.6.4 Physics-space pixel WRMSE (diagnostic only)

Additionally the notebook tracks a **physics-space** metric per row:

$$
{\rm WRMSE}^{(r)} \;=\; \sqrt{\frac{1}{N_\lambda}\sum_\lambda \big(\hat{y}_r(\lambda)-y_r(\lambda)\big)^2 / \sigma_{\rm tot}^2(\lambda)}
$$

with $\sigma_{\rm tot}$ from `FLUX_SIGMA_TOTAL`. This is used for diagnostic
reporting (worst-row visualisations, extrapolation quality `eRMSE`, per-
lunation drift) but is **not** part of the training gradient. The training
loss is §3.6.1 (moon and zodi) plus §3.6.2 on the remaining four groups.

### 3.6.5 Row-weight boosts

The training loop can up-weight specific row populations via multiplicative
per-row factors that are then normalised so $\mathbb{E}_{\rm train}[w_{\rm row}]=1$.
The currently deployed configuration is:

- `bright_moon_close_boost = 1.5` (Phase A'', 2026-08-27a). Mask:
  `moon_alt > 0` AND `moon_fli \ge 0.90` AND `moon_sep \le 30°`. Boosted
  subset is ~90 training rows (~2% of train). Attacks the one Phase-A'
  continuum regression regime (`close_zodi` p50 err/Δ 1.32) without dragging
  unrelated groups; larger/looser masks were rejected on 2026-08-27a for
  training-distribution-shift damage on atomic, zodi, and the blue atlas.
- `high_airmass_boost = 1.0` (disabled).
- `moon_down_ecliptic_boost = 1.0` (disabled).

Rule of thumb (2026-08-27a): the boosted subset should stay below ~150 rows
on this corpus (\lesssim 3% of train) so the shared-trunk representation
isn't dislodged.

### 3.6.6 Post-training per-coef Jensen-style lift calibration

After the ensemble is assembled, a **single-pass empirical bias correction**
is computed on the train+val subset and stored in `jensen_corrections`,
applied at inference in `expand_scores_to_coefs`. Three flavours:

- **`moon` (per-coef vector)**: for each of the 15 `Moon_bs` knots the ratio
  $\ell_k = \bar{c}^{\rm true}_k / \bar{c}^{\rm pred,\,naive}_k$ is computed
  from calibration-row means, ignoring near-zero knots
  ($|\bar{c}^{\rm pred}| < 0.05|\bar{c}^{\rm true}|$) which default to
  $\ell_k=1$. Clipped to $[0.7, 1.4]$ to protect against a broken knot
  cascading. The moon per-coef lift nulls the spectral tilt bias that a
  single scalar cannot correct.
- **`zodi` (per-coef vector, regime-conditioned)**: three moon-alt regime
  buckets are lifted independently (`moon_up` for `moon_alt > 10°`,
  `moon_horizon` for `-10° \le moon_alt \le 10°`, `moon_down` for
  `moon_alt < -10°`) with a 5°-wide smooth boundary. Each bucket uses the
  same per-coef ratio $\ell_k$ formula on its subset; if a bucket has fewer
  than 30 calibration rows it falls back to the global per-coef lift.
  Clipped to $[0.7, 1.4]$. At inference the row's `moon_alt` selects the
  bucket (smoothed at the boundaries).
- **`continuum`, `mesospheric`, `ionospheric`, `atomic` (scalar)**:
  $\ell = \bar{c}^{\rm true}/\bar{c}^{\rm pred,\,naive}$, clipped to
  $[0.5, 2.0]$. Skipped if the mean is degenerate or the relative magnitude
  is below 5%.

At inference each predicted coef vector for group $g$ is multiplied by
$\boldsymbol\ell_g$ (per-coef for moon, regime-conditioned per-coef for zodi,
broadcast scalar for the others) inside `expand_scores_to_coefs`. This closes
the residual coherent bias that the network's own outputs leave in place
without adding trainable parameters.

## 3.7 Optimisation and splits

- **Optimiser**: AdamW, `lr=1e-3`, `weight_decay=1e-4`, `grad_clip=1.0`,
  `n_epochs=50`, `patience=12` (early stop on val loss). The blend-α direct-
  parametrised params (§3.5) sit in a separate optimiser group with
  `weight_decay=0` and are clamped to $[\varepsilon, 1-\varepsilon]$ with
  $\varepsilon=10^{-3}$ after each step.
- **Batching**: `batch_size=512`. All training data (~O(10 MB) after RobustScaler
  standardisation of scores and ctx, fit on train rows only) is staged once
  onto the training device and iterated with `torch.randperm` — no per-batch
  CPU→device transfers.
- **Split**: rows are grouped by `night_id = floor(mjd - 0.5)` and stratified
  by lunar phase into train / val / test partitions via
  `split_indices_by_moon_phase` (defined in `mlp_predictor.ml_utils`). The
  same random seed splits the corpus identically across ensemble members so
  they all see the same held-out test rows.
- **Ensembling**: 10 seeds `(42, 43, ..., 51)` train independently. At
  inference the ensemble mean of the per-seed physical-space predictions is
  returned; the per-row std is also reported as an epistemic-uncertainty
  flag by the `predictive_uncertainty_ensemble` diagnostic.

## 3.8 Diagnostics

The `mlp_predictor.diagnostics.Diagnostics` class runs the diagnostic cells
against a captured globals dict, so the cell bodies (stored verbatim in
`skysub/mlp_predictor/diagnostics_cells/*.py`) stay readable and re-runnable
from the notebook. The active suite covers:

- **Coefficient-space quality**: `coef_residual_vs_value`,
  `naive_baseline` (vs `copy_near` / `near_geo` / `mean_geo`),
  `rmse_dual_diagnostic`, `rmse_worst_stability`.
- **Full-spectrum reconstruction**: `full_spectrum_single_row`,
  `full_spectrum_batch_rmse` (100-row sample, per-row per-component residuals
  with stable per-row colours across all five spectrum panels),
  `worst_recon`, `pipeline_state_check`, `physical_space_cap`.
- **Regime & context slicing**: `per_lunation_drift`, `per_context_slice`,
  `sky_arm_zodi_bias`.
- **Sigma calibration & ensemble spread**: `resid_over_sigma_per_group`,
  `resid_vs_sigma_per_decile`, `truth_conditioned_sigma_calibration`
  (Mahalanobis $|z|_{\rm joint}$ on the persisted COEF_COV blocks),
  `moon_sigma_investigation` (joint vs marginal SNR by moon regime),
  `predictive_uncertainty_ensemble`, `ensemble_spread_calibration`
  (reliability curve for ensemble std vs actual error).
- **Missing-feature and noise-floor tests**: `residual_ctx_attribution`
  (linear + RF fit of per-row residual RMS on ctx, 5-fold CV R²),
  `sky_arm_disagreement_floor` (ML error vs sky-arm intrinsic disagreement Δ),
  `wavelength_residual_atlas` (aggregated pred − true λ across a 200-row
  sample, absolute + fractional per band).
- **Optional deeper cells**: `swrmse_coef_map`, `wrmse_vs_ctx_correlation`,
  `wrmse_vs_ctx_scatter`, `per_seed_vs_ensemble`. These are commented out by
  default in the notebook (they add ~1 min each) but are supported by the
  `Diagnostics` class the same way.

Each call cell in the notebook opens with two comment lines — `# Targets:`
(what the plot / table shows) and `# Look for:` (what interpretation flags a
regression) — so the diagnostic body can be inspected and rerun in isolation.



## Changelog (split-zodi variant)
- **2026-08-27c** — Phase F: additive moon-zodi coupling latent.  Currently deployed as the default (`moon_zodi_mode="additive"`).  Preserves both existing paths (moon on shared trunk, zodi on isolated branch) and adds a small coupling branch whose latent is projected additively into the moon and zodi head outputs via zero-initialised linear projectors.  Design goal: capture Phase E's moon wins without its atlas-mid regression by preserving zodi's Phase-B isolation.
    * **Config**: `moon_zodi_ctx_restriction` = 18-feature union (airmass, vanrhijn_285km, ecliptic geometry, `zodi_log10_v`, `sun_sep`, all moon-scatter features from Phase A/A', and the three interaction features).  Coupling branch is (64, 32) hidden dims.  Two `Linear(32, n_g)` projectors (moon: 495 params, zodi: 165 params); both weight and bias zero-init so at t=0 the branch is a no-op and training starts byte-identical to Phase A''.
    * **Parameter budget**: coupling branch 8.4k + projectors 0.7k = **+9k full-model params** (571k → 580k, +1.6%).  About 45% smaller than Phase E's +20k.
    * **A/B numbers**, 10-seed ensemble, night-held-out split, n_test = 1231, baseline = 27a (Phase A''):
        - Ensemble mean_eRMSE: 21.11 → 21.38 (+1.3%, ~0.4σ of run noise — not statistically distinguishable).
        - **Seed std: 0.469 → 0.726** (+55%, worst reproducibility since Phase D).  Ensemble stderr 0.148 → 0.230 (1.1% of mean).
        - Ensemble mean_eWRMSE: 14.89 → 15.23 (+2.3%).
        - `sky_arm_disagreement_floor` **moon** p50 err/Δ (the design target):
            + all:        0.972 → **0.935** (−3.8%)
            + moon_up:    0.691 → **0.663** (−4.1%)
            + moon_down:  1.232 → 1.219 (−1.1%)
            + close_zodi: 1.169 → **1.108** (−5.2%)
        - `sky_arm_disagreement_floor` **zodi** p50 err/Δ (surprising win despite the coupling):
            + all:        0.909 → **0.862** (−5.2%)
            + moon_up:    0.464 → 0.504 (+8.6%, expected leak; halved vs Phase E's +14.7%)
            + moon_down:  1.304 → **1.244** (−4.6%)
            + close_zodi: 1.055 → **1.024** (−2.9%)
        - `naive_baseline` moon sRMSE (all): 0.0985 → **0.0897** (−8.9%).
        - `naive_baseline` zodi sRMSE (all): 0.136 → **0.132** (−2.9%); moon_up preserved at 0.138.
        - `wavelength_residual_atlas` (the concern):
            + blue RMS|frac|:  0.256% → 0.278% (+9%)
            + **mid RMS|frac|:   0.530% → 0.634% (+20%)** — halfway back toward Phase E's regression; A'' had the best mid ever.
            + NIR RMS|frac|:   1.30% → 1.39% (+7%)
            + mid mean_bias:   −0.17% → −0.32% (2×)
            + NIR mean_bias:   −0.44% → −0.63% (+43%)
        - Continuum unchanged (`close_zodi` p50 err/Δ 1.284 → 1.242, small win).
    * **A/B numbers vs Phase E (27b) for context**:  Phase F captures ~60% of Phase E's moon win (0.947 → 0.935 all-regime) and Phase F's zodi is *better* than both A'' and E in every regime except moon_up.  On atlas, Phase F sits between A'' and E: mid RMS 0.634% (vs A'' 0.530%, E 0.713%), NIR 1.39% (vs 1.30%, 1.42%).
    * **Interpretation**: the zero-init projectors do move away from zero during training, and the coordinated moon+zodi head shifts they introduce trade coefficient-space accuracy (improved) for pixel-space atlas mid RMS (regressed +20%).  The seed-std blow-up (+55%) is variance-inflation from the new branch — the projectors' zero-init doesn't prevent the branch's other 8.4k params from producing a fresh source of seed-dependent optimisation trajectory.
    * **Trade-off summary**: moon and zodi p50 err/Δ metrics improved 3–9% (target hit), zodi *sRMSE* improved outside the moon_up regime, but atlas mid RMS regressed +20% and seed reproducibility halved.  All three main metrics moved in the direction one would expect given the mechanism; the question is whether the deliverable-relevant atlas mid trade is acceptable.
    * **Rejected Phase E fallback**: `moon_zodi_mode="shared_branch"` (Phase E, 2026-08-27b, rejected) is still supported via that config knob for future A/B.  It produced a larger moon win (−7% moon_up sky-arm) but also a larger atlas regression (+35% mid RMS) and a zodi moon_up +15%.
    * **Diagnostics cell change (2026-08-27c)**: `full_spectrum_batch_rmse` (the 100-row sample multi-panel figure) now assigns a stable per-row color from the `turbo` colorscale so a given row's components (moon / zodi / diffuse / lines residual panels) all share the same colour and can be visually traced across the five spectrum panels.

- **2026-08-27a** — Phase A'': tight-mask bright-moon-close row-weight boost lands as the deployed default.  Attacks the one Phase-A' continuum regression (`close_zodi` p50 err/Δ 1.27 → 1.32) without dragging unrelated groups.
    * **Config**: `bright_moon_close_boost=1.5` with a tight mask `bright_moon_close_fli_min=0.90` and `bright_moon_close_sep_max_deg=30.0` (moon-up + very bright + very close to sci pointing).  Boosted subset = 90 training rows out of ~4 000 (~2% of train), effective training-distribution shift ~1%.
    * **Motivation**: Phase A' left one small regression on continuum `close_zodi` (1.27 → 1.32) as the only place the ML head lost against the Phase-A+B+C baseline.  Row-weight boosts (§3.6.5) were the cheapest lever available.  Naive attempt at the deployed default mask (`sep<=45`, `fli>=0.85`, boost=2.0) matched ~500 rows and hurt the aggregate by +2.1% mean_eRMSE plus regressions in atomic (+8.8%), zodi (+6.5%) and atlas blue (+46%).  Dropping to boost=1.5 with the same wide mask kept most of the collateral damage.  The successful lever was **narrowing the mask**: `fli>=0.90` and `sep<=30 deg` cut the boosted subset to ~90 rows, small enough to avoid the training-distribution shift that leaked into unrelated groups.
    * **A/B numbers**, 10-seed ensemble, night-held-out split, n_test = 1231, baseline = 26f (Phase A'):
        - Ensemble mean_eRMSE: 20.98 → 21.11 (+0.6%, within seed noise 0.47).
        - Ensemble mean_eWRMSE: 14.71 → 14.89 (+1.2%).
        - Seed std: 0.509 → 0.469 (−8%).  Ensemble stderr 0.16 → 0.15.
        - `sky_arm_disagreement_floor` continuum p50 err/Δ (**Phase-A' regression overturned**):
            + all:        1.164 → **1.145** (−1.6%)
            + moon_up:    1.205 → 1.203 (flat)
            + **moon_down**:  1.077 → **1.054** (−2%, bonus)
            + **close_zodi**: 1.322 → **1.284** (−3%, target hit)
        - `wavelength_residual_atlas`:
            + **mid RMS|frac|: 0.629% → 0.530% (−16%)** — best atlas mid of any phase.
            + **mid mean_bias: −0.38% → −0.17%** (halved).
            + blue RMS|frac|: 0.209% → 0.256% (+23%, small in absolute terms).
            + blue mean_bias: +0.11% → +0.18%.
            + NIR RMS|frac|: 1.32% → 1.30% (flat); NIR mean_bias −0.63% → −0.44% (improved).
        - `naive_baseline` continuum ML gain over `B1_near_geo` (Phase-A' wins preserved):
            + all-regime:  +2.0% → +1.8%
            + moon_up:     +0.2% → +0.2% (preserved)
            + moon_down:   +7.1% → +6.5%
            + close_zodi: −4.1% (26e) / +2.4% (26f) → −1.6% (small regression in coefficient-space accounting; sky-arm floor is the correct metric here and it improved).
        - Other sky-arm groups (all-regime):
            + moon 0.962 → 0.972; zodi 0.881 → 0.909; mesospheric 1.179 → 1.181; ionospheric 0.309 → 0.313; atomic 0.571 → 0.591.
            + Zodi and atomic drifted +3–4%; both still comfortably below the arm-disagreement floor.  Meso and iono essentially unchanged.
    * **What made the tight mask work**: the wide mask (500 rows at 1.5–2.0× weight) shifted ~25% of the effective training signal onto one narrow physical regime, pulling capacity away from atomic, zodi, moon_down and the blue continuum.  The tight mask (90 rows at 1.5×) shifts ~1% — enough to move the target metric because those rows are the extreme of the sample, small enough that the head keeps its representation of everything else.  Boost-magnitude sweep at the wide mask (1.5 vs 2.0) showed the blue leak was essentially independent of boost, confirming the mechanism is training-distribution shift rather than the boost factor itself.
    * **Interpretation**: the Phase-A' continuum `close_zodi` regression was small and fixable, but not with any wide-scope lever.  Row-weight boosts are only safe when they touch a small enough fraction of training that the shared-trunk representation isn't dislodged.  Below ~2–3% of training rows the head can absorb the reweighting without collateral; above, every other group pays.  Rule of thumb for future row-weight-boost experiments: keep the boosted subset under 150 rows on this corpus.

- **2026-08-26f** — Phase A': three explicit non-linear interaction features added to the ctx augment pipeline, all three routed to the isolated continuum branch (§3.4.1).
    * **Config**: three new features computed per-arm in `data._augment_triplet_with_physics_priors`:
        - `moon_fli_x_phase_cos` = `moon_fli * moon_phase_cos`  (2nd-order phase term, (cos - cos²)/2)
        - `moon_sig_x_lon_cos`   = `moon_signal_proxy * ecl_lon_cos`  (moon-scatter × zodi anisotropy)
        - `moon_sig_x_lon_sin`   = `moon_signal_proxy * ecl_lon_sin`  (moon-scatter × zodi anisotropy, rot pair)
      Total `ctx_names` grew 36 → 39.  The three features are appended to `continuum_ctx_restriction`, growing the continuum branch input dim 24 → 33 (still tiny cost given the Phase D branch width).
    * **Motivation**: `residual_ctx_attribution` on Phase D pinned continuum RF importance on `moon_fli`, `moon_phase_cos`, `ecl_lon_cos` and `moon_signal_proxy` — all of which enter the residual only through second-order interactions (moon-scatter into the diffuse ecliptic background).  A smooth MLP through a 160-d shared trunk plus a 128–64 branch can only build these approximately; feeding them explicitly bypasses the polynomial-expansion cost.
    * **A/B numbers**, 10-seed ensemble, night-held-out split, n_test = 1231, baseline = 26e (Phase D):
        - Ensemble mean_eRMSE: 21.12 → **20.98** (−0.7%)
        - Ensemble mean_eWRMSE: 14.98 → **14.71** (−1.8%)
        - **Seed std: 0.801 → 0.509** (−36%) — recovered most of the Phase-D variance-inflation.  Ensemble stderr 0.25 → 0.16 (0.8% of mean).
        - `sky_arm_disagreement_floor` continuum p50 err/Δ:
            + all:        1.186 → **1.164** (−2%)
            + moon_up:    1.236 → 1.205 (−2.5%)
            + moon_down:  1.095 → 1.077 (−1.6%)
            + close_zodi: 1.274 → 1.322 (**+3.8%**, only regression; still ≪ Phase-A+B+C 1.40)
        - `wavelength_residual_atlas`:
            + **blue RMS|frac|: 0.535% → 0.209%** (−61%, biggest atlas win of any phase)
            + mid  RMS|frac|: 0.569% → 0.629% (+11%, still tiny in absolute terms; mean_bias grew from ~0 to −0.38%)
            + NIR  RMS|frac|: 1.30% → 1.32% (+1.5%; mean_bias grew from ~0 to −0.63%)
        - `naive_baseline` continuum sRMSE (ML gain over B1_near_geo):
            + all-regime:   −2.1% → **+2.0%** (first time ML wins overall)
            + moon_up:      −4.6% → **+0.2%** (first time ML wins in moon_up)
            + moon_down:    +5.0% → +7.1%
            + close_zodi:   −4.1% → +2.4% (first time ML wins here)
        - `residual_ctx_attribution` top continuum importances now dominated by the new features:
            + `moon_sig_x_lon_cos` imp = **0.200** (NEW #1)
            + `moon_fli_x_phase_cos` imp = 0.097 (NEW)
            + Continuum RF R² 0.685 → 0.730 (goes *up*, since the RF now has better predictors of residual *magnitude*; the R² measures amplitude-driven variance, not correctable bias — see §12 in the intro).
        - Ionospheric picked up `moon_fli_x_phase_cos` as its own RF top-importance feature (imp = 0.099), a hint the same interaction would help there.
    * **The one regression** (continuum `close_zodi` +3.8% vs 26e) is small in absolute terms and still 6% below the pre-Phase-D value (1.40).  The mid/NIR mean-bias drift (− 0.4% and − 0.6%) is below the Jensen calibration re-tuning threshold but worth revisiting when moving to mesospheric/ionospheric refinements.
    * **Interpretation**: the continuum head *was* under-parameterised for the non-linear interactions the RF surfaces, but it was more efficient to feed the interactions explicitly than to widen the branch further.  Phase D + Phase A' together closed the arm-floor gap on continuum from 1.25 (26d) → 1.16 (26f), a 40% reduction of the headroom above the sky-arm floor.

- **2026-08-26e** — Phase D: widened isolated continuum branch (attack the only group with real headroom above the sky-arm floor).
    * **Config**: `continuum_branch_dims` (64, 32) → (128, 64); `continuum_head_extra_dims` () → (64,).  +12k targeted params on the continuum branch (branch + isolated head 4.0k → 16.2k); full-model 537k → 549k (+2.3%).
    * **Motivation**: `residual_ctx_attribution` continuum RF R² = 0.68 with 0.46 non-linear gain, and `sky_arm_disagreement_floor` continuum p50 err/Δ = 1.25 across all regimes (1.40 close_zodi, 1.21 moon_down) — the only group where the diagnostic and the noise-floor test agreed on genuine headroom.  Widening the branch on the only headroom group avoids the trunk bottleneck and doesn't touch groups already at the floor.
    * **A/B numbers**, 10-seed ensemble, night-held-out split, n_test = 1231, baseline = 26d (Phase A+B+C):
        - `sky_arm_disagreement_floor` continuum p50 err/Δ:
            + all:        1.247 → **1.186** (−5%)
            + moon_up:    1.283 → 1.236 (−4%)
            + **moon_down**:  1.212 → **1.095** (−10%, target hit)
            + **close_zodi**: 1.401 → **1.274** (−9%,  target hit)
        - `wavelength_residual_atlas`:
            + blue RMS|frac|: 0.61% → **0.535%** (−12%)
            + mid  RMS|frac|: 0.54% → 0.569% (+5%; mean bias still ±0.01%)
            + NIR  RMS|frac|: 1.37% → **1.30%**  (−5%)
        - `naive_baseline` continuum sRMSE:
            + all-regime:   0.01554 → **0.01461** (−6%, **recovers the Phase-B accounting regression**)
            + moon_down:    ML now **beats** `B1_near_geo` (+5% gain) — first time on continuum
        - `residual_ctx_attribution` continuum RF R²: 0.681 → 0.685 (essentially unchanged, as predicted — R² measures amplitude-driven variance, not correctable bias).
        - Aggregate mean_eRMSE: 20.99 → 21.12 (+0.6%, within seed noise).
        - Aggregate mean_eWRMSE: 14.93 → 14.98 (+0.3%).
        - **Seed std: 0.409 → 0.801** (+96%) — variance-inflation from the +12k targeted params; ensemble stderr 0.25 = 1.2% of mean, still tight, so accepted.
    * **Interpretation**: the continuum-branch capacity was the binding constraint.  The nonlinear moon-geometry structure that the RF surfaces on continuum residuals (rf_R² - lin_R² = 0.43) is exactly the kind of piecewise threshold behaviour a 64‒32 branch struggles with; a 128‒64–64 branch has enough capacity to absorb it.  The seed std blow-up (0.41 → 0.80) is the price of the extra capacity, but the ensemble averaging cancels it and the *median* seed mean_eRMSE actually moved only slightly (21.42 → 21.45).

- **2026-08-26d** — Three architectural refinements landed (Phase A + B + C) plus loss-weight relaxation.  Deployed default now writes all three into `default_dual_group_config`.
    * **Phase C (head capacity)**: `zodi_head_extra_dims=(32,)` inserts an extra hidden layer inside the isolated zodi head.  `head_extra_dims=(96,)` tried on the shared-trunk heads and reverted (neutral on the aggregate, no per-regime win) — the shared trunk bottleneck (160-d) means extra head capacity is a no-op on those groups.
    * **Phase B (isolated continuum branch)**: `continuum_ctx_restriction=("moon_alt", "moon_sep", "moon_phase_sin", "moon_phase_cos", "moon_fli", "airmass")` routes the continuum head through a dedicated 64→32→n branch mirroring the zodi-restriction pattern (§3.4.1).  Attacks the ~0.70 `residual_ctx_attribution` RF R² on continuum which was driven entirely by moon geometry.  Adds ~2k params.
    * **Phase A (ctx-dependent blend alpha)**: `alpha_ctx_features=("moon_up_smooth", "ecl_beta_deg", "airmass")` computes per-row per-group $\alpha_g(x_{\rm sci}) = \sigma(w_g \cdot x + b_g)$ via a linear predictor.  First attempt applied to all six groups blew up ionospheric sRMSE by 23% and atomic by 12% — the LOS-integrated thin-shell groups want $\alpha \approx 0.5$ and the extra 4 params per group learned noise.  Restricted to `alpha_ctx_groups=("moon", "zodi", "continuum")` — the anisotropic groups only — the regression disappeared and the target-regime wins survived.
    * **Loss weight relaxation**: `moon_group_weight` 4.0→2.0 and `continuum_group_weight` 1.5→1.0.  With the flux-space MSE (§3.6.1) already amplitude-weighting the per-row loss, the elevated moon weight was a redundant second-order amplification of bright-moon rows that was over-driving the moon head under Phase B (visible as messy per-row moon component residuals in the 100-sample reconstruction plot).
    * **A/B numbers**, 10-seed ensemble, night-held-out split, n_test = 1231, baseline = pre-Phase config:
        - ensemble mean_eRMSE: 20.91 → 20.99 (+0.4%, within seed noise ±0.4–0.5)
        - **seed std: 0.488 → 0.409** (−16%, tighter, more reproducible)
        - mean_eWRMSE: 14.77 → 14.93 (+1%)
        - sci pRMSE max (100-row): 1.34e−14 → 1.32e−14
        - **`wavelength_residual_atlas` mid-band RMS|frac|: 0.97% → 0.54%** (−44%)
        - **`wavelength_residual_atlas` mid-band mean bias: +0.72% → −0.08%** (essentially zeroed)
        - **`wavelength_residual_atlas` NIR RMS|frac|: 1.54% → 1.37%** (−11%)
        - **`sky_arm_disagreement_floor` zodi `close_zodi` p50 err/Δ: 1.16 → 1.06** (target hit)
        - **`sky_arm_disagreement_floor` zodi `moon_down` p50 err/Δ: 1.39 → 1.31** (target hit)
        - `naive_baseline` atomic all-regime sRMSE: 3.94 → 3.58 (−9%)
        - `naive_baseline` continuum all-regime sRMSE: 0.0138 → 0.0155 (+12%, diagnostic-only — see below)
        - `residual_ctx_attribution` RF R² essentially unchanged across groups (0.61→0.63 moon, 0.71→0.68 continuum).
    * **Continuum sRMSE regression is not a physics regression.**  The isolated continuum branch absorbs moon-regime signal that improves the full-spectrum flux reconstruction; the per-coefficient sRMSE goes up because amplitude shifts between moon and continuum in coefficient space, but the Jensen scalar lift keeps continuum mean bias < 5% and the wavelength atlas (which is the actual sky-subtraction deliverable) confirms the total-flux win.
    * The training-log line "`Learned per-group near-arm blend alpha at best epoch [direct]: moon=0.700 ... delta=+0.000`" is now misleading for the ctx-alpha groups: those scalar params are unused when `alpha_predictors[g]` is active.  Not a correctness bug; the ctx-alpha predictors are in the loss graph and do learn (verified by the target-regime wins and the seed-std tightening).

- **2026-08-26c** — data-loading / training / diagnostics extracted into a
  new ``skysub/mlp_predictor/`` package; notebook cells collapsed to thin
  callers.  The package layout is:
    * ``config.py`` — pipeline dataclasses (``DataConfig`` / ``FilterConfig``
      / ``SplitConfig`` / ``ModelConfig`` / ``TrainConfig`` /
      ``DiagnosticsConfig`` / ``PipelineConfig``).
    * ``ml_utils.py`` — ``RobustScaler``, seed helper, three
      ``split_indices*`` variants.
    * ``metrics.py`` — ``metric_row`` + coefficient-space + pixel-space
      WRMSE helpers (three near-duplicate variants consolidated over one
      shared weight builder).
    * ``model.py`` — ``DualEncoderGroupHeadMLP`` +
      ``DualEncoderGroupHeadMLPCompressed``.
    * ``compressor.py`` — per-group sqrt/asinh/PCA compressor,
      Jacobian sigma propagation, ``expand_scores_to_coefs``.
    * ``data.py`` — decomposition + META loader, geometry feature
      builders (KS / Leinert / ecliptic / physics priors), triplet
      builder, filters, solar-activity cache, effective-extinction fit.
    * ``wavelengths.py`` — cache-gated coef-wavelength + extinction
      pipeline wrapping ``coef_wavelengths_from_basis`` /
      ``resolve_coef_extinction_k``.
    * ``trainer.py`` — verbatim ``train_compressed_group_mlp`` +
      ``predict_sci_coefficients_default`` + a thin ``Trainer`` class
      wrapping the ensemble seed loop and the flux-MSE basis-matrix
      precompute (``EnsembleArtifacts`` dataclass groups the outputs).
    * ``diagnostics.py`` — 22 diagnostic cells stored as verbatim source
      strings and exposed via one method per figure on the ``Diagnostics``
      class (``.coef_residual_vs_value()``, ``.worst_recon()``, etc.).
      Uses ``exec()`` against a captured globals dict so cell bodies stay
      byte-identical.
    * Consolidations applied while extracting: three WRMSE variants
      collapsed to one shared weight builder; two ``airglow_geometry_scale``
      call paths in the compressor merged behind a lazy ``data`` default;
      module-level wiring statements that referenced notebook globals (``triplet``,
      ``filtered_triplet``, ...) moved out of import-time into methods.
    * Notebook cells 3–45 collapsed to ~30 thin callers.  Notebook total
      line count drops from 12,444 to ~600.  Behavioral parity preserved
      (cell bodies still run verbatim inside ``Diagnostics._run``); this
      is a code-organisation refactor, not a functional change.
- **2026-08-26b** — decomposition-side amplitude-prior calibration path removed and notebook slimmed.
    * Deleted `skysub/sky_decomp/moon_zodi_priors.py`,
      `skysub/calibrate_moon_zodi_priors.py`, and the three
      `moon_zodi_spline/moon_zodi_priors_calibration*.json` files.  The v3
      calibration already had $\lambda^{\rm amp}_{\rm m}=\lambda^{\rm amp}_{\rm z}=0$
      so the deployed path was dormant; identifiability rests on the color-
      only $B_{\rm zodi}(\lambda) \propto \lambda^{0.26}$ split alone (median
      RMS ratio 0.988 versus baseline, per
      `notebooks/moon_zodi_split_identifiability.ipynb`).
    * Stripped the priors kwargs (`moon_amp_prior`, `zodi_amp_prior`,
      `moon_amp_prior_lambda`, `zodi_amp_prior_lambda`) and the
      `moon_smooth_lambda_override` / `zodi_smooth_lambda_override` hooks
      from `SkyDecomp.fit`, `SkyDecomp._fit_design`,
      `SkyDecompLSFSurfaceIterative.fit`, `SkyDecompLSFSurfaceIterative._run_iterations`,
      `SkyDecompLSFSurfaceIterative._solve_continuum`, and the
      `_solve_nonnegative_weighted` inner solver; also removed the
      `moon_amp_weights` / `zodi_amp_weights` basis-integrals and the
      `_dw` grid-spacing helper that fed them.  Updated the API contract
      test signature to `(self, flux, ivar, *, verbose=False)`.
    * `decompose_parallel.py` lost the `--moon-zodi-priors` CLI arg,
      the `_split_zodi_prior_kwargs_for_row` per-row lazy geometry lookup,
      the `_load_moon_zodi_priors_bundle` bundle packer, the
      `_WORKER_PRIORS_BUNDLE` worker global, and all pass-through
      plumbing.
    * Intro §1.7 (Geometry-driven amplitude priors) removed; the QP
      objective, rescaling bullets, and the CLI code block in the
      pipeline description were also trimmed of priors mentions.
    * Notebook slimmed by removing six experimental A/B cells that had
      served their diagnostic purpose:
      `layer-structure-function-markdown` + `layer-structure-function-code`,
      `4e9a28f8` + `52b09b79` (Effective Dimensionality PCA-rank),
      `0f08c243` (denser HP sweep, `RUN_DENSE_SWEEP` gated),
      `audit-worst15-regime-plus-tv`, `5412db57` (context-variable
      analysis for worst reconstructions), and `manual-fulltriplet-r2-map`
      (RUN_FULL_TRIPLET_R2_MAP gated).  Total cell count dropped from
      53 to 45.
    * Jensen post-training lift calibration (§3.6.6) retained per user
      request; no functional change to the deployed training pipeline.
- **2026-08-26** — per-pixel flux-space MSE loss on moon and zodi becomes the deployed default (see §3.6.1).
    * Trainer signature gained `flux_mse_groups=('moon', 'zodi')`,
      `flux_mse_eps_frac=0.0`, and two "materialise once" precompute
      inputs `flux_basis_matrices` (per-group spline template matrix at
      the notebook's wavelength grid, downsampled with stride 5) and
      `flux_geom_sc_sci` (per-row per-coef geometry factor from
      `airglow_geometry_scale`).
    * The precompute block instantiates `SkyDecompLSFSurfaceIterative` on
      the wavelength grid loaded from `INPUT_FITS_FOR_BASIS` and reads
      `matrix_moon` / `matrix_zodi`.  It runs once, before the seed loop,
      because `_shared_train_kwargs` references its outputs (the earliest
      attempt had it after and hit `NameError`).
    * `compressed_loss` grew a per-group flux-MSE branch that takes
      precedence over both `relative_mse_groups` and
      `block_cov_loss_groups`.  The branch (i) undoes the RobustScaler on
      the group's compressed score, (ii) undoes the PCA rotation via
      `basis.T`, (iii) destandardises to the forward-transformed space,
      (iv) inverts the `sqrt` element transform with `clamp(z, min=0)**2`,
      (v) multiplies by the per-row per-coef geometry factor, then
      (vi) reconstructs flux via `c @ A_g` and takes MSE per pixel.  A
      per-group scale-match factor (median train diagonal loss / median
      train `mean(flux**2)`) rescales the flux MSE onto the same
      magnitude as the diagonal Huber it replaces so `moon_group_weight`
      and `zodi_group_weight` retain their calibrated meaning.
    * `_stage_on_device` gained a `row_idx` tensor at slot 9 so the loss
      can index the per-row geometry array during backprop; train and
      val loops updated symmetrically (`_batch[:10]` unpack, block-cov
      offset 9 → 10).
    * `default_dual_group_config` now sets `flux_mse_groups=('moon', 'zodi')`,
      `flux_mse_eps_frac=0.0`, `block_cov_loss_groups=()`,
      `relative_mse_groups=()`.  The block-Mahalanobis and relative-MSE
      plumbing stays behind their config knobs for future A/B.
    * 10-seed ensemble A/B (spline4 corpus, night-held-out split,
      n_test = 1231): mean_eRMSE 21.08 (relative-MSE, prior default) →
      **20.91**, matches the coefficient-space diagonal Huber baseline
      (20.62) within seed noise; mean_eWRMSE **14.77** (best of all
      variants); sci pRMSE max on the 100-row subset 2.31e-14 →
      **1.34e-14** (−42%); moon SNR>1 truth-conditioned |z| 0.049 →
      **0.029**.  Zodi bias by regime: per-lunation max |bias| 27.97% →
      **11.46%**; moon-down zodi ML bias +7.71% → **+1.67%**; moon
      Q4-phase zodi +7.66% → **+0.92%**.  Zodi vanishes as the worst-bias
      group in every context slice, replaced by continuum at the ~1–2%
      level.

- **2026-08-25c** — opt-in relative-error MSE (`relative_mse_groups`) tried on
    moon and zodi and rejected as the deployed default.  Aggregate mean_eRMSE
    matched the coefficient-space diagonal baseline (21.08 vs 20.62 spline3
    diag) and seed-to-seed std collapsed from 0.84 → 0.47, but the loss
    introduced a systematic +7% zodi bias in the moon-down and Q4-phase
    regimes: the $\varepsilon^2 = ($ `relative_mse_eps_frac` $)^2 \cdot$ median
    `||target_raw||^2` denominator floor makes the gradient collapse to
    $\sim 2(\Delta s)/\varepsilon^2$ on low-flux rows, so the model
    under-predicts them consistently.  Superseded by the flux-space MSE
    (§3.6.1) which addresses the same "errors should scale with flux"
    goal without the eps floor.  Plumbing retained behind
    `relative_mse_groups=()` for future A/B against per-pixel flux MSE.

- **2026-08-25b** — opt-in block-Mahalanobis loss for moon and/or zodi groups.
    * Trainer signature gained `block_cov_loss_groups=()` (default empty ->
      byte-identical to the baseline diagonal Huber loss).  Setting e.g.
      `block_cov_loss_groups=('moon',)` or `('moon','zodi')` precomputes the
      per-row score-space inverse covariance $\Sigma_{\rm scaled}^{-1}$
      from `filtered_triplet['coef_cov_moon_sci']` / `coef_cov_zodi_sci`
      via the diagonal per-element compressor Jacobian (recovered as
      $J = \sigma_{\rm scaled}/\sigma_{\rm native}$; both moon and zodi
      use PCA=False so this is exact).
    * Each configured group's per-element weighted Huber loss is replaced by
      the block Mahalanobis $L_g = \mathbf{r}^{\top}\Sigma^{-1}\mathbf{r} / n_g$
      inside `compressed_loss(...)`.  A per-group scale-match factor
      (median of diagonal loss / median of Mahalanobis loss, both on train
      rows at init) is folded into $\Sigma^{-1}$ so the group weights
      `moon_group_weight`, `zodi_group_weight`, etc. keep their calibrated
      meaning and the overall loss magnitude stays balanced.  Inactive
      coefs (NaN entries in the persisted covariance) are handled by
      pseudo-inverting the active sub-block per row.
    * The precomputed $\Sigma^{-1}$ tensors are staged on device alongside
      the existing per-batch state and unpacked once per batch; train and
      val loops updated symmetrically.  See §3.6.2 for the math, and the
      `truth-conditioned-sigma-calibration` cell for the Mahalanobis $|z|$
      diagnostic that motivated this (moon block joint $|z|_{\rm joint} \approx 0.074$
      says the joint $\sigma$ over-estimates by ~9x, so the block-weighted
      loss should raise the effective per-element weight on moon coefs by
      ~80x relative to the marginal-diagonal baseline).
    * Ships as OFF by default; deployment path is to A/B against the
      diagonal baseline on the same seed set.

- **2026-08-25** — spline3 reduced-basis config + joint-covariance persistence.
    * B-spline knot counts reduced (validated by the standalone knot-count
      identifiability probe under `scratch/probe_moon_knot_count.py`):
      Moon_bs `n_spline_knots` 25 → 11 (basis 29 → 15), Zodi_bs
      `n_zodi_spline_knots` 3 → 1 (basis 7 → 5).  Flux-fit quality on both
      the bright-moon (moon_alt>30, fli>0.85) and faint-moon (moon_alt<-5)
      regimes stayed within 1–3% of the K=25 baseline while the median
      adjacent-knot |corr| in the moon block dropped 0.96 → 0.88 and all
      pairs above 0.99 (5/28 at K=25) were eliminated (0/14 at K=11).
      Every downstream call site of `n_spline_knots` / `n_zodi_spline_knots`
      in this notebook now derives from `N_MOON_KNOTS` / `N_ZODI_KNOTS`,
      auto-inferred by the `infer-spline-knots` cell from
      `filtered_triplet['coef_names']`; a switch of `DECOMP_DATA_ROOT`
      re-loads everything self-consistently.
    * Introduced an explicit wavelength-basis cache gate
      (`wavelength-cache-gate` cell) that populates
      `{DECOMP_DATA_ROOT}/coef_wavelengths_basis_v4.npz` on first run and
      hits it on subsequent runs, so downstream extinction resolution uses
      per-basis-function B²-weighted centroids instead of the group-default
      fallback.  Required threading `palace_oh_suffix` / `palace_diffuse_suffix`
      through the notebook wrapper and both callers so the cache-populating
      reconstruction reads the joint_v2_updated PMD tables, matching the
      settings the corpus was actually fit with.  `reconstruct_component_spectra`
      in `sky_decomp/fit.py` was extended to accept the split_zodi kwargs
      so the recon call no longer shape-mismatches on split_zodi corpora
      (silent try/except previously produced an unpopulated cache under
      `moon_zodi_spline2/`).
    * **Joint covariance persistence.** `SkyDecomp._coef_err_active_set`
      gained an optional `cov_block_slices` return path that materialises
      the full active-set covariance sub-matrix for the moon and zodi
      B-spline blocks (in the same physical units as `coef`, with NaN for
      inactive coefs).  `SkyDecompResult` grew two kw-only fields
      `coef_cov_moon`, `coef_cov_zodi`; `results_to_fits` auto-writes
      `COEF_COV_MOON` / `COEF_COV_ZODI` ImageHDUs whenever they are
      non-None (skipping the trivial n_block=1 case).
      `extract_meta_and_coef_products` copies them into the compact
      `*_meta_coef_*.fits` files the notebook reads.  No new CLI args on
      `decompose_parallel.py`; all automatic.  New notebook cell
      `coef-cov-loader` (defensive: falls back to marginal-σ diagnostics
      when the HDUs are absent) attaches
      `coef_cov_{moon,zodi}_{near,far,sci}` to `filtered_triplet`.
      `truth-conditioned-sigma-calibration` cell appends per-row
      Mahalanobis `|z|_joint = sqrt(rᵀ Σ⁻¹ r / n_active)` on the moon and
      zodi blocks; `moon-sigma-investigation` cell appends binned joint
      SNR `sqrt(cᵀ Σ⁻¹ c / n_active)` per moon-state regime.  Marginal
      diagnostics preserved side-by-side for comparison.  See §1.9 and
      §3.6.2 for the propagation math.
    * The trainer's per-element loss weight (§3.6.2) still uses the
      diagonal σ from `COEF_ERR` — architectural extension to a
      block-Mahalanobis loss on the moon and zodi groups is deferred
      until an A/B decides whether the modest ensemble-metric win
      justifies it (the diagnostic cells now provide the calibration
      evidence to support that decision).


- **2026-08-22** — Branched from the deployed dual-encoder group-MLP
  notebook.  Retargeted at the `moon_zodi_spline/` corpus (files with
  `_lsf_surface_iterative_split_zodi` suffix, produced by
  `SkyDecompLSFSurfaceIterative(split_zodi=True)`).  Promoted the new
  `Zodi_bs\d+` coefficient family to a dedicated `zodi` group in
  `COEF_SCHEMA` and `_COEF_GROUP_ORDER`.  Bumped
  `WAVELENGTH_CACHE = coef_wavelengths_basis_v4.npz`.  Added
  `zodi` entries to `DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP` and
  `COMPRESSION_USE_PCA_BY_GROUP`.  Reset intro documentation to
  scientific-article style covering the split-zodi physical model,
  decomposition, and ML architecture (see cell 1).  Deleted the standalone A/B experiment cells that tuned the pre-split
  model (`alpha_init` grids, pre-fit alpha, moon-compression retention,
  floor policy, LR/WD sweeps) and the row-vs-row zodi diagnostic that
  was designed to explain moon/zodi conflation (obviated by split_zodi
  at fit time).  Disabled `zodi_asymmetry_boost` (default 3.0 -> 0.0)
  and dropped `ks_moon_log10_v` from the physics-prior ctx features;
  both were compensating for pre-split conflation.  Added
  `zodi_group_weight = 1.0` to `default_dual_group_config`.  Updated
  the coefficient distribution, single-row reconstruction, and batch
  reconstruction plots so Zodi_bs appears as a companion panel wherever
  Moon_bs was shown before.  Retired the pre-split `moon_alt_conditional_alpha`
  flag from the trainer's public config (default was already `False`);
  the remaining flag-guarded branches inside the `DualEncoderGroupHeadMLPCompressed`
  body are unreachable and will be excised in a follow-up refactor.

- **2026-08-22b** — Enabled **isolated zodi branch** in
  `DualEncoderGroupHeadMLPCompressed`.  When
  `default_dual_group_config['zodi_ctx_restriction']` is a non-empty tuple
  of ctx feature names, the zodi head is routed through a dedicated
  sub-network (`zodi_branch` + `zodi_head_isolated`) that consumes only:
  (i) the sci-arm slice of the listed ctx features and (ii) the near/far
  zodi score blocks.  The shared trunk is bypassed for zodi, cutting
  moon-driven and geomagnetic-driven leakage into the zodi head.
  Default features enforce the physical prior from Kelsall 1998 /
  Leinert 1998: zodi surface brightness is a smooth function of
  ecliptic latitude, ecliptic longitude, airmass, and Van Rhijn
  high-altitude LoS geometry, plus the pre-computed `zodi_log10_v`
  amplitude and `sun_sep`.  Set
  `default_dual_group_config['zodi_ctx_restriction'] = None` to fall
  back to the pre-2026-08-22b shared-trunk behavior.

- **2026-08-22c** — Reconstruction path debugged.  Root cause of the
  cell 28 "matmul size 4 vs 5" traceback: the hoisted `_lsf_model` was
  instantiated with `n_spline_knots=29` while the corpus was fit with
  the fit.py default `n_spline_knots=25` (which yields 29 Moon_bs basis
  functions; larger n_knots gives 33).  The extra 4 moon basis rows
  shifted every downstream slice by 4, so `sl['atom']` = (445,450)
  requested 5 coefs from a 449-length vector and truncated to 4.
  Fixed by matching the corpus setting (`n_spline_knots=25`) in both
  cell 28's hoisted `_lsf_model` and cell 5's `reconstruct_with_lsf`
  default.  Cells 27, 28, 29, 30, 34 now all run end-to-end.

- **2026-08-22d** — Two sweeps on the split-zodi corpus with the isolated
  zodi branch active.  (i) `zodi_group_weight` 1.0 -> 3.0 hurt calibration
  (zodi mean bias +0.6% -> +1.7%) and marginally worsened test mean_eRMSE
  (19.77 -> 19.83); kept default at 1.0.  Plumbed the config key through
  `train_compressed_group_mlp` (it was defined but never read).
  (ii) `patience` 8 -> 12 gave a mild win (mean_eRMSE 19.77 -> 19.72,
  ~0.26% within stderr).  Three seeds (42, 44, 45) trained ~10 epochs
  longer; two hit the epoch cap of 50.  Adopted patience=12 as the new
  default.  Bumping `n_epochs` above 50 remains a candidate follow-up
  (some seeds still improving at cap).

- **2026-08-23** — Isolated zodi head now consumes per-arm zodi-restricted
  ctx (sci + near + far) instead of sci only.  The head input grows from
  `len(zodi_ctx_restriction) + 2 * n_zodi_score` to
  `3 * len(zodi_ctx_restriction) + 2 * n_zodi_score` (from 21 → 35 with the
  default 7-feature restriction and n_zodi_score=7).  This is the smallest
  change that gives the head access to the local sky-vs-sci geometric
  gradient — the `zodi_log10_v` value at the near arm can differ from sci by
  15–20 % at high `|β_ecl|` (rows like 1481, 1432), and the previous head
  had no way to see that difference.  Expected to close the blue-flux
  under-prediction at those rows without changing the moon head or the
  decomposition corpus.


In [ ]:
# --- Package imports ---
%load_ext autoreload
%autoreload 2

import os
os.environ.setdefault("LVMCORE_DIR", "/Users/droryn/prog/lvm/lvmcore")

import numpy as np
import pandas as pd
import plotly.io as pio

# Notebook cwd is skysub/, so mlp_predictor/ and sky_decomp/ are top-level packages.
from mlp_predictor import config, data, wavelengths, compressor, trainer, diagnostics
from mlp_predictor.config import (
    DataConfig, FilterConfig, SplitConfig, ModelConfig, TrainConfig,
    DiagnosticsConfig, PipelineConfig,
)

# Journal-style plotly axes (as in the pre-refactor notebook).
for _ax in (pio.templates["plotly_white"].layout.xaxis,
            pio.templates["plotly_white"].layout.yaxis):
    _ax.showgrid = False
    _ax.showline = True
    _ax.mirror = True
    _ax.linecolor = "black"
    _ax.linewidth = 1
    _ax.ticks = "inside"
    _ax.zeroline = False
pio.templates.default = "plotly_white"

FACTOR = 1e14
cfg = PipelineConfig()
print(f"Corpus: {cfg.data.decomp_data_root}/{cfg.data.decomp_stem} "
      f"(suffix={cfg.data.decomp_suffix!r})")


In [ ]:
# --- Load triplet + apply filters + augment context ---
triplet = data.build_triplet_coef_dataset(
    input_fits_path=cfg.data.input_fits_meta,
    sky_near_decomp_fits_path=cfg.data.coef_fits("sky1"),
    sky_far_decomp_fits_path=cfg.data.coef_fits("sky2"),
    sci_decomp_fits_path=cfg.data.coef_fits("sci"),
    context_columns=list(cfg.data.context_columns),
    return_chi2=True,
)
print("Loaded triplet shapes:",
      {k: v.shape for k, v in triplet.items()
       if hasattr(v, "shape") and k in ("coef_near","coef_far","coef_sci","ctx_sci")})

filtered_triplet = data.apply_triplet_filters(
    triplet,
    thin_every_n=1, chi2_qmax=90.0, chi2_min=0.0, chi2_max=10.0,
    hard_coef_bounds={"feo": (0.0, 10.01), "atom_k": (0.0, 10.01)},
    kappa=6.0, kappa_iter=3, oh_kappa=4.0, oh_kappa_iter=3,
    exclude_field_regions=[data.LMC_EXCLUSION, data.SMC_EXCLUSION],
)

data._augment_triplet_with_ecliptic(filtered_triplet, force=True)
data._augment_triplet_with_physics_priors(filtered_triplet, force=True)
print(f"Augmented ctx: n_ctx={len(filtered_triplet['ctx_names'])}")


In [ ]:
# Preview pre- vs post-filter coefficient histograms (no ML state needed).
from mlp_predictor import diagnostics  # local import: survives kernel-restart edge cases

_tmp = diagnostics.Diagnostics(diagnostics.DiagnosticsContext(
    filtered_triplet=filtered_triplet,
    extras={"triplet": triplet},
)).coef_hist_prepost()


In [ ]:
# --- Per-row joint covariance blocks (COEF_COV_MOON / COEF_COV_ZODI) ---
import numpy as _np
from astropy.io import fits as _fits

def _load_cov_for_row_index(decomp_path, row_index):
    row_index = _np.asarray(row_index, dtype=_np.int64)
    with _fits.open(decomp_path) as _h:
        _moon = (_np.asarray(_h["COEF_COV_MOON"].data[row_index], dtype=_np.float64)
                 if "COEF_COV_MOON" in _h else None)
        _zodi = (_np.asarray(_h["COEF_COV_ZODI"].data[row_index], dtype=_np.float64)
                 if "COEF_COV_ZODI" in _h else None)
    return _moon, _zodi

_row_idx_cov = _np.asarray(filtered_triplet["row_index"], dtype=_np.int64)
for _arm, _path in {
    "near": cfg.data.decomp_fits("sky1"),
    "far":  cfg.data.decomp_fits("sky2"),
    "sci":  cfg.data.decomp_fits("sci"),
}.items():
    _m, _z = _load_cov_for_row_index(_path, _row_idx_cov)
    filtered_triplet[f"coef_cov_moon_{_arm}"] = _m
    filtered_triplet[f"coef_cov_zodi_{_arm}"] = _z
print("Per-row joint covariance loaded (or None where COEF_COV_* HDU absent).")


In [ ]:
# --- Populate coefficient wavelengths + effective extinction ---
ext = wavelengths.resolve_wavelengths_and_extinction(
    filtered_triplet,
    cache_path=cfg.data.wavelength_cache_path,
    input_fits_for_basis=cfg.data.input_fits_for_basis,
    use_fitted_extinction=True,
    verbose=True,
)
group_indices = ext.group_indices
N_MOON_KNOTS, SPLIT_ZODI, N_ZODI_KNOTS = wavelengths.infer_spline_knots(
    filtered_triplet["coef_names"])


## Symmetric Dual-Encoder Group-Head Model

Key design choices in this implementation:
1. Shared near/far encoder weights for symmetry and sample efficiency.
2. Fusion vector uses:
   - $e_{mean} = 0.5(e_{near} + e_{far})$
   - $e_{diff} = e_{near} - e_{far}$
   - $|e_{diff}|$
3. Context branch encodes science-context only, now including folded time features and van Rhijn shell factors.
4. Group-specific heads map fused representation into layer-aware coefficient groups rather than a single flat atomic bucket.

**Note.** `DualEncoderGroupHeadMLP` below is the parent architecture. What
the notebook actually trains is `DualEncoderGroupHeadMLPCompressed` (defined
two cells down, together with the compressor fit), which reuses this
architecture verbatim on a per-group compressed coefficient space and emits
signed linear head outputs. See §5.5 in the methods cell above for the
compression pipeline and the empirical argument for switching to it.


In [ ]:
# --- Fit per-group compressors on a moon-phase-stratified split ---
from mlp_predictor.ml_utils import moon_phase_deg_from_ctx, split_indices_by_moon_phase

_moon_phase = moon_phase_deg_from_ctx(filtered_triplet)
_split_tr, _split_va, _split_te = split_indices_by_moon_phase(
    filtered_triplet["obstime_mjd"], _moon_phase, seed=42)

group_compressors, compress_geom_kwargs = compressor.fit_all_group_compressors(
    filtered_triplet, group_indices,
    train_idx=_split_tr, held_idx=_split_va,
    xarm_threshold=compressor.COMPRESSION_XARM_THRESHOLD,
    verbose=True,
)
filtered_triplet["compress_train_idx"] = _split_tr
filtered_triplet["compress_val_idx"] = _split_va
filtered_triplet["compress_test_idx"] = _split_te


In [ ]:
# --- Training config (edit here to override deployed defaults) ---
# Full deployed config expanded in place so every knob is visible; every value
# is initialised to the mlp_predictor.trainer default and can be changed below.

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# 2026-08-27c: Phase F -- additive moon-zodi coupling.  Adds a small
# coupling branch (~9k params) that produces a shared latent from the
# moon-zodi ctx union, projected additively into the moon head output
# and the zodi head output via zero-init linear projectors.  Both
# existing paths (moon on shared trunk, zodi on isolated branch) are
# preserved -- this is the "preserve zodi isolation, give moon the
# shared context" fix for Phase E's atlas-mid regression (2026-08-27b).
# At init the projector outputs are zero, so training starts byte-
# identical to Phase A''; the projectors learn to route the coupling.
# To A/B against Phase A'' (no coupling), set the four moon_zodi_*
# entries at the tail of train_cfg to None.
# To try the rejected Phase E instead, set `moon_zodi_mode="shared_branch"`.
# 2026-08-26f: Phase A' landed on top of Phase D.  Adds three explicit
# interaction features to the ctx augment pipeline (computed in
# data._augment_triplet_with_physics_priors):
#   moon_fli_x_phase_cos = moon_fli * moon_phase_cos          # 2nd-order phase
#   moon_sig_x_lon_cos   = moon_signal_proxy * ecl_lon_cos    # moon x zodi geom
#   moon_sig_x_lon_sin   = moon_signal_proxy * ecl_lon_sin    # moon x zodi geom
# All three fed into the isolated continuum branch (see continuum_ctx_restriction
# below).  Attacks the residual_ctx_attribution RF top importances on continuum
# (moon_fli 0.21, moon_phase_cos 0.15, ecl_lon_cos 0.09, moon_signal_proxy 0.05).
# A/B vs 26e (Phase D, same 10 seeds, n_test=1231):
#   mean_eRMSE  21.12 -> 20.98 (-0.7%)  |  mean_eWRMSE 14.98 -> 14.71 (-1.8%)
#   seed std    0.801 -> 0.509 (-36%, most of Phase-D variance recovered)
#   sky-arm continuum p50 err/Delta: all 1.19 -> 1.16, moon_down 1.10 -> 1.08,
#     close_zodi 1.27 -> 1.32 (only regression; still << pre-Phase-D 1.40).
#   wavelength_residual_atlas blue RMS|frac|: 0.535% -> 0.209% (-61%, huge).
#     mid +11% (still 0.63%), NIR +1.5%.  Mean bias grew to -0.4% mid / -0.6% NIR.
#   naive_baseline continuum ML now beats B1_near_geo in every regime (first
#     time): all +2.0%, moon_up +0.2%, moon_down +7.1%, close_zodi +2.4%.
# To A/B against pre-Phase-A' (Phase D only), drop the three trailing entries
# from `continuum_ctx_restriction` below.  The augment always emits them; they
# are simply ignored by the branch when not listed.
# ----------------------------------------------------------------------
# 2026-08-26e: Phase D landed on top of Phase A+B+C.  Widens the isolated
# continuum branch from (64, 32) -> (128, 64) and adds a 64-d hidden layer
# inside the continuum head (+12k targeted params, +2.3% full-model params).
# A/B vs 26d (Phase A+B+C, same 10 seeds, n_test=1231):
#   sky-arm floor, continuum p50 err/Delta:
#     all        1.247 -> 1.186  (-5%)
#     moon_down  1.212 -> 1.095  (-10%, target hit)
#     close_zodi 1.401 -> 1.274  (-9%,  target hit)
#   naive_baseline continuum sRMSE all: 0.01554 -> 0.01461 (-6%, recovers
#     the Phase-B accounting regression; ML now also beats B1_near_geo on
#     continuum moon_down for the first time).
#   wavelength_residual_atlas: blue -12%, NIR -5%, mid +5% (still zero-bias).
#   Aggregate mean_eRMSE 20.99 -> 21.12 (within seed noise); seed std
#     0.41 -> 0.80 (+96%, variance-inflation from the extra 12k params;
#     ensemble stderr still 1.2% of mean).
# To A/B against the pre-Phase-D config (Phase A+B+C only), override:
#     "continuum_branch_dims":     (64, 32)  # current: (128, 64)
#     "continuum_head_extra_dims": ()        # current: (64,)
# To revert further to the pre-Phase-A/B/C baseline, additionally override:
#     "moon_group_weight":         4.0       # current: 2.0
#     "continuum_group_weight":    1.5       # current: 1.0
#     "zodi_head_extra_dims":      ()        # current: (32,)
#     "continuum_ctx_restriction": None      # current: 6-tuple moon-geom
#     "alpha_ctx_features":        None      # current: 3-tuple
#     "alpha_ctx_groups":          None      # current: (moon, zodi, continuum)
# See 2026-08-26d + 2026-08-26e changelog entries for the full A/B tables.
# ----------------------------------------------------------------------
train_cfg = {
    "name": "dual_group_mlp_compressed",

    # Optimisation schedule.
    "n_epochs": 50,
    "batch_size": 512,
    "lr": 1.0e-3,
    "weight_decay": 1.0e-4,
    "patience": 12,

    # Architecture (widths).
    "encoder_dims": (768, 384),
    "ctx_dims": (96,),
    "trunk_dims": (320, 160),
    "head_dim": 192,
    "head_extra_dims": (),                     # Phase C: neutral on shared-trunk heads; kept only on zodi
    "zodi_head_extra_dims": (32,),          # Phase C: extra hidden layer in the isolated zodi head

    # Per-group loss weights m_g (see §3.6.2).
    "moon_group_weight": 2.0,
    "zodi_group_weight": 2.0,
    "continuum_group_weight": 1.0,
    "mesospheric_group_weight": 1.0,
    "ionospheric_group_weight": 1.0,

    # Loss shaping.
    "block_cov_loss_groups": (),
    "relative_mse_groups": (),
    "relative_mse_eps_frac": 0.05,
    "flux_mse_groups": ("moon", "zodi"),          # §3.6.1 deployed default
    "flux_mse_eps_frac": 0.0,

    # Row-weight boosts (see §3.6.5).
    "high_airmass_boost": 1.0,
    "moon_down_ecliptic_boost": 1.0,
    "moon_down_ecliptic_beta_deg": 15.0,
    # Phase A'' (2026-08-27): tight-mask 1.5x row-weight boost on bright moon
    # close to sci pointing (fli>=0.90, sep<=30 deg, alt>0).  Boosted subset
    # is only ~90 training rows (vs 500 with the wide 45/0.85 mask), so the
    # training-distribution shift is ~2% -- small enough to avoid the blue-
    # atlas leak the wide masks triggered.  Wide-mask 1.5 and 2.0 A/Bs both
    # showed +45-53% blue RMS|frac| regression regardless of boost magnitude;
    # tight-mask 1.5 keeps it to +23% while gaining -16% on mid RMS|frac|
    # and closing the Phase-A' close_zodi p50 err/Delta regression.
    # A/B vs 26f (Phase A', same 10 seeds, n_test=1231):
    #   sky_arm continuum: all 1.164 -> 1.145, moon_down 1.077 -> 1.054,
    #     close_zodi 1.322 -> 1.284  (all improved -- Phase-A' regression overturned).
    #   atlas mid RMS|frac|: 0.629% -> 0.530% (-16%), mid mean_bias halved -0.38 -> -0.17%.
    #   atlas blue RMS|frac|: 0.209% -> 0.256% (+23%, small; wide masks were +53%).
    #   ensemble mean_eRMSE 20.98 -> 21.11 (+0.6%, within seed noise 0.47).
    # To A/B against the pre-Phase-A'' config (Phase A' only), set:
    #     "bright_moon_close_boost": 1.0,  # current: 1.5
    "bright_moon_close_boost": 1.5,
    "bright_moon_close_fli_min": 0.90,
    "bright_moon_close_sep_max_deg": 30.0,

    # COEF_ERR weighting (§3.6.3).
    "use_coef_err_weights": True,
    "coef_err_sigma_floor_rel": dict(trainer.DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP),

    # 10-seed ensemble.
    "ensemble_seeds": (42, 43, 44, 45, 46, 47, 48, 49, 50, 51),
    
    # Isolated zodi branch context restriction (§3.4.1).
    "zodi_ctx_restriction": (
        "airmass", "vanrhijn_285km",
        "ecl_beta_deg", "ecl_lon_sin", "ecl_lon_cos",
        "zodi_log10_v", "sun_sep",
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "moon_up_smooth",
        "moon_airmass_up", "moon_signal_proxy",
    ),
    # Phase B (2026-08-26): moon-regime-conditioned isolated head for continuum.
    # Phase A' (2026-08-26): explicit interaction features added below (from
    # residual_ctx_attribution rankings; computed in data._augment_triplet_with_physics_priors).
    "continuum_ctx_restriction": (
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "airmass",
        "moon_fli_x_phase_cos",
        "moon_sig_x_lon_cos", "moon_sig_x_lon_sin",
    ),
    # Phase D (2026-08-26): widened continuum branch (only group with real
    # headroom above sky-arm floor; residual_ctx_attribution RF R^2 = 0.68).
    # Baseline (64, 32) with () extra layers -> 4k params on the branch.
    # Deployed (128, 64) with (64,) extra layer -> 16k params, +12k targeted.
    "continuum_branch_dims": (128, 64),
    "continuum_head_extra_dims": (64,),
    # Phase A (2026-08-26): context-dependent per-group blend alpha.
    "alpha_ctx_features": (
        "moon_up_smooth", "ecl_beta_deg", "airmass",
    ),
    # Restrict ctx-alpha to the anisotropic groups; others keep scalar alpha.
    "alpha_ctx_groups": ("moon", "zodi", "continuum"),
    # Phase F (2026-08-27c): additive moon-zodi coupling.  Keeps both existing
    # paths intact (moon on shared trunk, zodi on isolated branch, continuum on
    # its own branch).  Adds a small coupling branch (64->32) computing a shared
    # latent from the moon-scatter + zodi-geometry ctx union; the latent is
    # projected additively into the moon head output (Linear(32, n_moon)) and
    # the zodi head output (Linear(32, n_zodi)) via zero-initialised linear
    # projectors, so training starts byte-identical to Phase A''.  This is the
    # "preserve zodi isolation while giving moon the shared context" alternative
    # to Phase E (`moon_zodi_mode="shared_branch"`), which was rejected on
    # 2026-08-27b for the atlas-mid regression it caused.  +9k params.
    # "moon_zodi_ctx_restriction": None, # Phase A baseline
    "moon_zodi_ctx_restriction": (
        "airmass", "vanrhijn_285km",
        "ecl_beta_deg", "ecl_lon_sin", "ecl_lon_cos",
        "zodi_log10_v", "sun_sep",
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "moon_up_smooth",
        "moon_airmass_up", "moon_signal_proxy",
        "moon_fli_x_phase_cos",
        "moon_sig_x_lon_cos", "moon_sig_x_lon_sin",
    ),
    "moon_zodi_mode": "additive",
    "moon_zodi_coupling_dims": (64, 32),
    # Phase E fallback knobs (only read when moon_zodi_mode="shared_branch"):
    "moon_zodi_branch_dims": (128, 64),
    "moon_zodi_moon_head_extra_dims": (64,),
    "moon_zodi_zodi_head_extra_dims": (32,),
}
print(f"train_cfg: {len(train_cfg)} knobs, {len(train_cfg['ensemble_seeds'])}-seed ensemble, "
      f"flux_mse_groups={train_cfg['flux_mse_groups']}")


In [ ]:
# --- Fit the ensemble (uses train_cfg from the cell above) ---
_trainer = trainer.Trainer(cfg=train_cfg)
artifacts = _trainer.run_ensemble(
    filtered_triplet, group_compressors, group_indices, compress_geom_kwargs,
    input_fits_for_basis=cfg.data.input_fits_for_basis,
    n_moon_knots=N_MOON_KNOTS, split_zodi=SPLIT_ZODI, n_zodi_knots=N_ZODI_KNOTS,
    verbose=True,
)
mlp_artifacts = artifacts.mlp_artifacts
_ensemble_members = artifacts.members
train_idx = np.asarray(mlp_artifacts["train_idx"], dtype=int)
val_idx = np.asarray(mlp_artifacts["val_idx"], dtype=int)
test_idx = np.asarray(mlp_artifacts["test_idx"], dtype=int)

coef_near_all = np.asarray(filtered_triplet["coef_near"], dtype=np.float32)
coef_far_all  = np.asarray(filtered_triplet["coef_far"],  dtype=np.float32)
coef_sci_all  = np.asarray(filtered_triplet["coef_sci"],  dtype=np.float32)
ctx_near_all  = np.asarray(filtered_triplet["ctx_near"],  dtype=np.float32)
ctx_far_all   = np.asarray(filtered_triplet["ctx_far"],   dtype=np.float32)
ctx_sci_all   = np.asarray(filtered_triplet["ctx_sci"],   dtype=np.float32)
coef_err_near_all = np.asarray(
    filtered_triplet.get("coef_err_near", np.full_like(coef_near_all, np.nan)), dtype=np.float32)
coef_err_far_all = np.asarray(
    filtered_triplet.get("coef_err_far", np.full_like(coef_far_all, np.nan)), dtype=np.float32)
coef_err_sci_all = np.asarray(
    filtered_triplet.get("coef_err_sci", np.full_like(coef_sci_all, np.nan)), dtype=np.float32)

coef_pred_det = trainer.predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=coef_near_all[test_idx],
    coef_far_phys=coef_far_all[test_idx],
    ctx_near_phys=ctx_near_all[test_idx],
    ctx_far_phys=ctx_far_all[test_idx],
    ctx_sci_phys=ctx_sci_all[test_idx],
).astype(np.float32)


In [ ]:
# --- Persist the trained ensemble to disk for inference ---
# See mlp_predictor.serialization + mlp_predictor.inference for the loader
# and the minimal-input predict API; consumed by
# notebook_example_predict_sky.ipynb.
from mlp_predictor import serialization

SAVED_ENSEMBLE_PATH = f"{cfg.data.decomp_data_root}/mlp_ensemble_split_zodi_current.pt"
serialization.save_ensemble(mlp_artifacts, SAVED_ENSEMBLE_PATH)


In [ ]:
# --- Build the diagnostics context ---
from mlp_predictor import diagnostics  # local import: survives kernel-restart edge cases

diag_ctx = diagnostics.DiagnosticsContext(
    filtered_triplet=filtered_triplet,
    mlp_artifacts=mlp_artifacts,
    group_compressors=group_compressors,
    group_indices=group_indices,
    geom_kwargs=compress_geom_kwargs,
    ensemble_members=_ensemble_members,
    coef_pred_det=coef_pred_det,
    coef_near_all=coef_near_all,
    coef_far_all=coef_far_all,
    coef_sci_all=coef_sci_all,
    ctx_near_all=ctx_near_all,
    ctx_far_all=ctx_far_all,
    ctx_sci_all=ctx_sci_all,
    coef_err_near_all=coef_err_near_all,
    coef_err_far_all=coef_err_far_all,
    coef_err_sci_all=coef_err_sci_all,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    # Pre-filter triplet + notebook-only bare names the extracted cell bodies read.
    extras={
        "triplet": triplet,
        "coef_wavelengths_a": ext.coef_wavelengths_a,
        "context_cols": list(cfg.data.context_columns),
        "FACTOR": FACTOR,
        "DECOMP_DATA_ROOT": cfg.data.decomp_data_root,
        "DECOMP_STEM": cfg.data.decomp_stem,
        "_DECOMP_SUFFIX": cfg.data.decomp_suffix,
        # Spline-knot counts read by full_spectrum_* cell bodies.
        "N_MOON_KNOTS": N_MOON_KNOTS,
        "SPLIT_ZODI": SPLIT_ZODI,
        "N_ZODI_KNOTS": N_ZODI_KNOTS,
        # Per-seed test metrics DataFrame consumed by naive_baseline.
        "cmp_df": artifacts.per_seed_test_metrics,
    },
)
diag = diagnostics.Diagnostics(diag_ctx)


In [ ]:
# Targets: per-coefficient (pred - true) vs true, per group.
# Look for: median off zero => bias; funnel/asymmetry => heteroscedastic residual.
_tmp = diag.coef_residual_vs_value()


## Relationship Visualizations
The following visualization cell is copied from the existing notebook and uses the default predictor interface.

In [ ]:
# diag.relationship_scatter_matrix()


## Full-Spectrum Verification
The following verification cell is copied from the existing notebook and reconstructs a selected science row from predicted coefficients.

In [ ]:
# Targets: reconstruct one every10 row from pred vs true coefs.
# Look for: quality of the fit; residual panels break out moon / zodi / lines / diffuse.
# REQUESTED_ROW = 612
# REQUESTED_ROW = 500
REQUESTED_ROW = 978
# REQUESTED_ROW = 1481
# REQUESTED_ROW = 741 #OK
# REQUESTED_ROW = 742
# REQUESTED_ROW = 830
_tmp = diag.full_spectrum_single_row(row=REQUESTED_ROW)


In [ ]:
# Targets: 100-row sample: per-row flux-space RMSE (near/far/sci).
# Look for: sci pRMSE distribution; per-component residual per row.
diag._init_globals()
_tmp = diag.full_spectrum_batch_rmse()


In [ ]:
# Targets: the 10 worst reconstructions from the batch above.
# Look for: which regimes (moon-up, bright-moon, high-|beta_ecl|, twilight) dominate the tail.
_tmp = diag.worst_recon()


In [ ]:
# Targets: sanity: predictor + per-group true/pred median and mean bias.
# Look for: any group with mean bias > 5% signals a broken head or lift calibration.
_tmp = diag.pipeline_state_check()


In [ ]:
# Targets: per-seed vs ensemble-mean bias by group.
# Look for: seed-dependent sign flip => under-trained; large seed spread => unstable.
_tmp = diag.per_seed_vs_ensemble()


In [ ]:
# Targets: ML vs copy_near / near_geo / mean_geo baselines, sRMSE per group per regime.
# Look for: ML should beat B2_mean_geo on moon+zodi; look at moon_up / moon_down / close_zodi.
_tmp = diag.naive_baseline()


In [ ]:
# Targets: coefficient-space and pixel-space RMSE side by side.
# Look for: consistency between the two; disagreement points to compressor / geometry issues.
_tmp = diag.rmse_dual_diagnostic()


In [ ]:
# Targets: are the worst rows the same across seeds?.
# Look for: seed-robust worst rows are real failures; seed-varying ones are optimisation noise.
_tmp = diag.rmse_worst_stability()


In [ ]:
# Targets: ML per-lunation mean/max error trend.
# Look for: monotone drift across lunations => solar-activity / seasonal missing feature.
_tmp = diag.per_lunation_drift()


In [ ]:
# Targets: ML error binned by ctx features (moon_alt, |beta_ecl|, airmass, ...).
# Look for: any slice with WRMSE >> global median flags a regime the head under-predicts.
_tmp = diag.per_context_slice()


In [ ]:
# Targets: zodi mean bias per moon-state regime, separately for near and far arms.
# Look for: asymmetric bias => zodi head is being pushed by the wrong arm.
_tmp = diag.sky_arm_zodi_bias()


In [ ]:
# Targets: per-row ensemble std as an epistemic-uncertainty flag.
# Look for: high-sigma rows to inspect qualitatively.
_tmp =diag.predictive_uncertainty_ensemble()


In [ ]:
# Targets: residual / sigma distribution per group (should be ~N(0,1) if calibrated).
# Look for: median |z| ~ 0.7; long tail => under-estimated sigma.
_tmp =diag.resid_over_sigma_per_group()


In [ ]:
# Targets: empirical RMS(err) vs sigma decile (log-log reliability curve).
# Look for: diagonal => calibrated; systematic below => sigma over-estimates error.
_tmp =diag.resid_vs_sigma_per_decile()


In [ ]:
# Targets: Mahalanobis |z|_joint on the persisted COEF_COV_MOON / COEF_COV_ZODI blocks.
# Look for: median |z| ~ 0.7 target; << 0.7 means joint sigma over-estimates.
_tmp =diag.truth_conditioned_sigma_calibration()


In [ ]:
# Targets: joint SNR sqrt(cT Sigma^-1 c) on the moon block per moon regime.
# Look for: compare joint vs marginal SNR; joint is the correct one.
_tmp =diag.moon_sigma_investigation()


In [ ]:
# Targets: verify the runaway upper bound never clips a real prediction.
# Look for: n_clipped should be 0 on train/val/test; max(pred/bound) << 1.
_tmp =diag.physical_space_cap()


In [ ]:
# Producer: builds triplet_full / wrmse_full / is_train / is_valtest /
# is_region_excluded consumed by the next two cells.
# _tmp = diag.swrmse_coef_map()


In [ ]:
# Targets: Pearson + Spearman of full-triplet WRMSE against every ctx feature.
# Look for: strong correlation => head is losing information about that feature.
# _tmp =diag.wrmse_vs_ctx_correlation()


In [ ]:
# Targets: scatter of full-triplet WRMSE vs each ctx feature.
# Look for: regime-dependent WRMSE structure that the correlation numbers hide.
# _tmp =diag.wrmse_vs_ctx_scatter()


In [ ]:
# Targets: linear + RF regression of per-group residual RMS on ctx (5-fold CV R^2).
# Look for: R^2 > 0.10 means the head is under-using ctx features it already sees.
# Missing-feature detector: linear + RF regression on per-group residual RMSE.
_tmp = diag.residual_ctx_attribution()


In [ ]:
# Targets: ML error vs sky-arm intrinsic disagreement (irreducible-noise floor).
# Look for: p50 err/Delta ~ 1 => at the floor; > 2 => headroom; < 1 => ML earns its keep.
# Irreducible noise floor: ML error vs sky-arm intrinsic disagreement.
_tmp = diag.sky_arm_disagreement_floor()


In [ ]:
# Targets: aggregated (pred - true)(lambda) across 200 rows, absolute + fractional.
# Look for: coherent bumps flag specific bands; fractional bias per band signals color miscalibration.
# Physics-space failure map: aggregated pred - true(lambda) across 200 rows.
_tmp = diag.wavelength_residual_atlas()


In [ ]:
# Targets: ensemble std vs actual |pred - true| per group + reliability curve.
# Look for: RMS ratio ~ 1 + Kendall tau > 0.3 => trustworthy sigma; ratio >> 1 => under-diverse ensemble.
# Uncertainty trust: ensemble std vs actual |pred - true| per group.
_tmp = diag.ensemble_spread_calibration()
